<img src="https://cdn.uclouvain.be/groups/cms-editors-arec/charte-graphique-uclouvain/telechargements/logo_UCLouvain_format_jpg_quadri.jpg?itok=hCkd3jyK" alt="UCLouvain" width="220">

# LGEO2270 | Physical geography excursion
## WaTEM/SEDEM — NB01: Catchment baseline
**Condroz–Famenne · Soil erosion, runoff and landscape connectivity**

School of Geography · Earth and Life Institute · UCLouvain

### Model concept

![Conceptual diagram: geographic inputs, two model branches, soil management, buffer strips, infrastructure and scenario evaluation](https://raw.githubusercontent.com/kvanoost/Watem/main/assets/model_workflow.svg)

[Open full-size model diagram](https://github.com/kvanoost/Watem/blob/main/assets/model_workflow.svg)

The branches share the spatial domain and routing framework; event sediment production is erosivity-based, not dynamically coupled to runoff. Channel runoff uses explicit D8 links when enabled.

**How to use this notebook:** Run the seven numbered code blocks in order. Confirm both maps explicitly. All computations are local to the Colab runtime; no Google Drive mount. Retain the final ZIP for NB02 and NB03. This is a teaching model, not a calibrated forecast.

[Official UCLouvain logo and graphic charter](https://uclouvain.be/fr/presse/logo-de-l-uclouvain)

**Simulation design:** one annual erosion run and one 40 mm rainfall event per baseline or management scenario.


## 1. Start and select an approximate outlet
The map imagery is only a visual reference (Esri). Navigate to the site, click **Load rivers in current view** to retrieve the official SPW RHW vector network, then click and confirm an approximate outlet.

In [ ]:
#@title 1 — Initialize and choose approximate outlet
from pathlib import Path
import os, sys, shutil, subprocess, json, hashlib, time, platform
from datetime import datetime, timezone
ROOT = Path('/content/Watem_local')
DATA = ROOT / 'data/tutorial/langegracht'
OUTPUT = ROOT / 'outputs/student_preparation'
SOFTWARE = ROOT / 'software'
for p in (DATA.parent, OUTPUT, SOFTWARE): p.mkdir(parents=True, exist_ok=True)
REV = '226a7c06ae836f9736c1a88b74b5346033a2a8d6'
REPO = Path('/content/pywatemsedem')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
WORK = Path('/content/watem_condroz_runs') / RUN_ID
WORK.mkdir(parents=True)
VERBOSE=False #@param {type:'boolean'}
def run(cmd, **kwargs):
    args=list(map(str,cmd))
    if VERBOSE:
        print('+', ' '.join(args),flush=True)
    kwargs.setdefault('stdout',subprocess.PIPE)
    kwargs.setdefault('stderr',subprocess.STDOUT)
    kwargs.setdefault('text',True)
    p=subprocess.run(args,**kwargs)
    if VERBOSE or p.returncode:
        if p.stdout: print(p.stdout,flush=True)
        if p.stderr: print(p.stderr,flush=True)
    p.check_returncode()
    return p
if VERBOSE: print(ROOT, sys.version, platform.machine())
SITE_ID='my_catchment' #@param {type:'string'}
assert SITE_ID and SITE_ID.replace('_','').replace('-','').isalnum()
SITE=ROOT/'student_sites'/f'{SITE_ID}_{RUN_ID}'
assert not (SITE/'baseline'/'complete.json').exists(), 'Site already frozen: use another SITE_ID or go to NB02'
PACKAGE=SITE/'reference'
PACKAGE.mkdir(parents=True,exist_ok=True)
SEED=2271; RES=20; YEAR=2025
# local search window near Ciney; bounds chosen to stay in one DEM tile
SEARCH_LONLAT=(5.00,50.25,5.16,50.36)
TARGET_HA=250; TOL_HA=50
MIN_RIVER_CELLS=20 # >= 20 burned river pixels; not a physical length estimate
# Optional locally cached numerical SPW raster paths; do not use rendered WMS/MapServer imagery.
SPW_DEM=None
SPW_WALOUS=None # requires explicit class crosswalk; not implemented here
run([sys.executable,'-m','pip','install','pysheds','ipyleaflet','fiona'])

from google.colab import output
from IPython.display import display, HTML, JSON
import json, uuid, time
from shapely.geometry import shape, mapping, box, Point
import requests, unicodedata, math
selection={}
REST='https://geoservices.wallonie.be/arcgis/rest/services'
WMS='https://geoservices.wallonie.be/arcgis/services'
def service_json(path):
    r=requests.get(REST+'/'+path+'/MapServer',params={'f':'json'},timeout=45); r.raise_for_status()
    j=r.json(); assert 'layers' in j,j; return j
def norm(s):
    return ''.join(c for c in unicodedata.normalize('NFKD',s.lower()) if not unicodedata.combining(c))
print('Preparing map services…',flush=True)
picc=service_json('TOPOGRAPHIE/PICC_VDIFF')
def picc_id(*terms):
    matches=[l['id'] for l in picc['layers'] if l.get('subLayerIds') is None and all(t in norm(l['name']) for t in terms)]
    assert len(matches)==1,(terms,matches); return matches[0]
PICC_IDS=dict(river=picc_id('hydrographie','axe'),water=picc_id('hydrographie','emprise'),roads=picc_id('voirie','axe') if False else next(l['id'] for l in picc['layers'] if norm(l['name']).strip()=='voirie - axe'),buildings=picc_id('construction','batiment','emprise'))
RIVER_URL=REST+'/EAU/RES_HYDRO_WAL/MapServer/1'
if VERBOSE: print('Current PICC layer IDs:',PICC_IDS,'| river source: RHW axes')

# Browser-native SVG overlay, independent of the Colab widget bridge and basemap tiles.
token='watem_first_'+uuid.uuid4().hex
river_cache={}
def map_query(action,args):
    if action=='select':
        lon,lat=map(float,args)
        assert 2.8<lon<6.5 and 49.4<lat<51,'Choose a point in Wallonia'
        selection['geometry']=mapping(Point(lon,lat))
        return JSON({'ok':True})
    if action=='clear':
        selection.clear()
        return JSON({'ok':True})
    if action=='count':
        west,south,east,north=map(float,args)
        q=requests.get(RIVER_URL+'/query',params=dict(where='1=1',geometry=','.join(map(str,args)),geometryType='esriGeometryEnvelope',inSR=4326,spatialRel='esriSpatialRelIntersects',returnIdsOnly='true',f='json'),timeout=45)
        q.raise_for_status(); j=q.json()
        assert 'error' not in j,j
        ids=j.get('objectIds') or []
        key=uuid.uuid4().hex
        river_cache.clear(); river_cache[key]=ids
        return JSON({'key':key,'count':len(ids)})
    if action=='batch':
        key,start=args; ids=river_cache[key][int(start):int(start)+200]
        q=requests.get(RIVER_URL+'/query',params=dict(objectIds=','.join(map(str,ids)),outFields='OBJECTID',returnGeometry='true',outSR=4326,f='geojson',geometryPrecision=6),timeout=60)
        q.raise_for_status(); j=q.json()
        assert 'features' in j and not j.get('exceededTransferLimit'),str(j)[:250]
        assert len(j['features'])==len(ids),'Incomplete river batch'
        return JSON({'features':[dict(type='Feature',geometry=f['geometry'],properties={}) for f in j['features']]})
    raise ValueError(action)
output.register_callback(token,map_query)
html=r"""
<div id="TOKEN" style="font:14px system-ui;background:white;color:#222;padding:8px">
<h3 style="margin:0 0 8px">Choose an approximate outlet</h3>
<div style="display:flex;gap:10px;flex-wrap:wrap">
<select class="base"><option>Topographic</option><option>Satellite</option><option>Street</option></select>
<button class="refresh">Load rivers in current view</button>
<label><input class="show" type="checkbox" checked> Show rivers</label></div>
<div class="status" role="status" style="background:#fff2cc;padding:12px;margin:8px 0;font-weight:bold">Loading map library…</div>
<progress class="progress" max="1" style="width:100%;height:18px"></progress>
<div class="map" style="height:600px;background:#eee"></div>
<div class="selection" style="padding:10px">Click map to place approximate outlet, then confirm.</div>
<button class="confirm" disabled>Confirm approximate outlet</button>
</div>
<script>
(async()=>{
const root=document.getElementById('TOKEN'),q=s=>root.querySelector(s), status=q('.status'), progress=q('.progress');
function say(t,busy=true){status.textContent=t;status.style.background=busy?'#fff2cc':'#dcfce7';}
function kernel(action,args){return google.colab.kernel.invokeFunction('TOKEN',[action,args],{}).then(r=>{const d=r.data||{};if(!('application/json' in d))throw Error('No JSON response from Python: '+JSON.stringify(d).slice(0,250));return d['application/json'];});}
try{
const css=document.createElement('link');css.rel='stylesheet';css.href='https://unpkg.com/leaflet@1.9.4/dist/leaflet.css';document.head.appendChild(css);
if(!window.L)await new Promise((ok,fail)=>{const el=document.createElement('script');el.src='https://unpkg.com/leaflet@1.9.4/dist/leaflet.js';el.onload=ok;el.onerror=fail;document.head.appendChild(el);});
const map=L.map(q('.map'),{preferCanvas:false,maxZoom:20}).setView([50.36,4.9],12);
const esri='https://server.arcgisonline.com/ArcGIS/rest/services/';
const names={Topographic:'World_Topo_Map',Satellite:'World_Imagery',Street:'World_Street_Map'};
let base;function basemap(){if(base)base.remove();base=L.tileLayer(esri+names[q('.base').value]+'/MapServer/tile/{z}/{y}/{x}',{maxNativeZoom:19,maxZoom:20,attribution:'Esri'}).addTo(map);}
basemap();q('.base').onchange=basemap;
map.createPane('rivers');map.getPane('rivers').style.zIndex=450;map.getPane('rivers').style.pointerEvents='none';
const renderer=L.svg({pane:'rivers',padding:.5});renderer.addTo(map);
const rivers=L.geoJSON(null,{pane:'rivers',renderer,interactive:false,style:{color:'#00bfff',weight:3,opacity:1}}).addTo(map);
let busy=false,pending=false,loaded=null,timer;
function extent(){const b=map.getBounds().pad(.15);return [b.getWest(),b.getSouth(),b.getEast(),b.getNorth()];}
function covered(){const b=map.getBounds();return loaded&&loaded[0]<=b.getWest()&&loaded[1]<=b.getSouth()&&loaded[2]>=b.getEast()&&loaded[3]>=b.getNorth();}
async function refresh(force=false){
 if(busy){pending=true;return;}if(!force&&covered())return;
 busy=true;q('.refresh').disabled=true;progress.removeAttribute('value');say('1/3 Checking river features in visible map…');
 const ext=extent();
 try{
 const result=await kernel('count',ext),total=result.count;
 if(total>20000)throw Error(total.toLocaleString()+' features: zoom in slightly (limit 20,000).');
 progress.max=Math.max(1,total);progress.value=0;
 say('2/3 Downloading '+total.toLocaleString()+' river features…');
 rivers.clearLayers();
 for(let i=0;i<total;i+=200){
 const batch=await kernel('batch',[result.key,i]);
 rivers.addData({type:'FeatureCollection',features:batch.features});
 progress.value=Math.min(i+200,total);
 say('2/3 Loading rivers: '+progress.value.toLocaleString()+' / '+total.toLocaleString());
 await new Promise(requestAnimationFrame);
 }
 say('3/3 Drawing river overlay…');
 map.invalidateSize();await new Promise(requestAnimationFrame);
 loaded=ext;progress.value=progress.max;say('✓ '+total.toLocaleString()+' RHW features loaded and attached to map.',false);
 }catch(e){say('River loading error: '+e);status.style.background='#fee2e2';loaded=null;}
 finally{busy=false;q('.refresh').disabled=false;pending=false;}
}
q('.refresh').onclick=()=>refresh(true);
q('.show').onchange=()=>{if(q('.show').checked){rivers.addTo(map);}else rivers.remove();};
map.on('moveend',()=>{if(!busy&&!covered())say('Map extent changed. Click Load rivers in current view to retrieve rivers.',false);});

let marker=null,selected=null,revision=0,queue=Promise.resolve();
function pick(ll){revision++;selected=ll;q('.confirm').disabled=true;q('.confirm').textContent='Confirm approximate outlet';q('.selection').textContent='Selected: '+ll.lat.toFixed(5)+', '+ll.lng.toFixed(5)+' — confirm before continuing.';
 if(!marker){marker=L.marker(ll,{draggable:true}).addTo(map);marker.on('dragend',()=>pick(marker.getLatLng()));}else marker.setLatLng(ll);
 const rev=revision;queue=queue.catch(()=>{}).then(()=>kernel('clear',[]));queue.then(()=>{if(rev===revision)q('.confirm').disabled=false;}).catch(e=>q('.selection').textContent='Selection error: '+e);}
map.on('click',e=>pick(e.latlng));
q('.confirm').onclick=async()=>{if(!selected)return;const rev=revision,ll=selected;q('.confirm').disabled=true;try{await queue; if(rev!==revision)return;queue=kernel('select',[ll.lng,ll.lat]);await queue;if(rev===revision){q('.selection').textContent='✓ APPROXIMATE OUTLET CONFIRMED. Run the next cell.';q('.confirm').textContent='✓ Confirmed';}}catch(e){q('.selection').textContent='Confirmation error: '+e;q('.confirm').disabled=false;}};
L.control.scale().addTo(map);
if(window.ResizeObserver)new ResizeObserver(()=>map.invalidateSize()).observe(q('.map'));
say('Map ready. Zoom to your study area, then click Load rivers in current view.',false);
requestAnimationFrame(()=>{map.invalidateSize();});
}catch(e){say('Map initialization failed: '+e);status.style.background='#fee2e2';}
})();
</script>
"""
display(HTML(html.replace('TOKEN',token)))


## 2. Download terrain and prepare hydrology
Software: pinned pywatemsedem, WaTEM/SEDEM 5.0.3 and SAGA-WaTEM. No tutorial geographical files or Git LFS downloads are required. Model settings are embedded as a local teaching preset; an exact checksum-verified cached Langegracht INI is used if available. The preset is based on the upstream Molenbeek example and wrapper tests and is not verified identical to Langegracht.

| Web input | Source resolution / type | Conversion |
|---|---|---|
| Copernicus GLO-30 elevation | ~30 m | Bilinear resampling to 20 m Belgian Lambert 72; optional supplied SPW DEM |
| ESA WorldCover 2021 v200 | 10 m | Nearest-neighbour resampling to common grid |
| SPW RHW river axes | Vector | Rasterize for stream burning; condition DEM with SAGA sink filling |

The 20 m grid does not imply 20 m original elevation detail. D8 flow directions and contributing area are then computed from the conditioned DEM. A 2 km terrain buffer reduces edge effects.

Software diagnostics are hidden by default; enable VERBOSE in block 1 to see them. Final effective settings are displayed in block 5, not the initial template.

Block 2 reports five timed stages. Software is reused within the same runtime when the pinned version is available. A new Colab runtime still needs installation; remote raster and river services and the initial D8 numerical compilation can be slow. Debug output is off by default.

Terrain coverage: a fixed 12 km margin surrounds the initial search area (approximately 30 × 30 km for a point). No student buffer setting is required. The outlet map checks upstream coverage and area before confirmation. Select yellow or orange channels; grey channels have incomplete coverage and red channels exceed the 5,000 ha classroom limit.

In [ ]:
#@title 2 — Install, download and condition terrain
from shapely.geometry import shape, box
import math
assert 'geometry' in selection,'Place a marker or draw a polygon first'
selected=shape(selection['geometry']); OUTLET_POINT=selected if selected.geom_type=='Point' else None
if OUTLET_POINT is not None:
    # 6 km initial search area plus a fixed 12 km terrain buffer.
    dy=3/111; dx=dy/math.cos(math.radians(selected.y))
    AOI=box(selected.x-dx,selected.y-dy,selected.x+dx,selected.y+dy)
else: AOI=selected
SEARCH_LONLAT=AOI.bounds
assert AOI.is_valid and AOI.area>0
assert 2.8<SEARCH_LONLAT[0]<SEARCH_LONLAT[2]<6.5 and 49.4<SEARCH_LONLAT[1]<SEARCH_LONLAT[3]<51,'Select a small area within Wallonia'
assert SEARCH_LONLAT[2]-SEARCH_LONLAT[0]<.25 and SEARCH_LONLAT[3]-SEARCH_LONLAT[1]<.20
SNAP_MAX_M=100
MAX_CATCHMENT_HA=5000  # internal teaching safety cap, not a student setting
RECOMMENDED_CATCHMENT_HA=1000
TERRAIN_BUFFER_STEPS=(12,)  # fixed generous processing margin, km on each side
MIN_CATCHMENT_HA=50
assert MAX_CATCHMENT_HA>=MIN_CATCHMENT_HA
if VERBOSE: print('Terrain bounds:',SEARCH_LONLAT)
from time import perf_counter
_block_start=perf_counter()
_stage_start=_block_start
def stage(message):
    global _stage_start
    _stage_start=perf_counter()
    print(message,flush=True)
def done(message):
    print(f'  ✓ {message} ({perf_counter()-_stage_start:.1f} s)',flush=True)
stage('1/5 · Checking model software')


import urllib.request
os_release = dict(line.strip().split('=',1) for line in Path('/etc/os-release').read_text().splitlines() if '=' in line)
codename = os_release.get('VERSION_CODENAME','').strip('"')
assert platform.machine() == 'x86_64', 'Expected x86_64 Colab runtime'
assert codename in {'jammy','noble','focal','bookworm'}, f'Unsupported OS for pinned SAGA release: {codename}'
if not shutil.which('saga_cmd'):
    run(['apt-get','update','-qq'])
    run(['apt-get','install','-y','saga'])
def download(url, path):
    if not path.exists():
        tmp = path.with_suffix(path.suffix+'.part')
        try:
            with urllib.request.urlopen(url,timeout=120) as response, tmp.open('wb') as f:
                shutil.copyfileobj(response,f)
            tmp.replace(path)
        finally:
            tmp.unlink(missing_ok=True)
    return path
saga_name = f'saga-vpo_2025.6.11-1-ge95f136.{codename}_amd64.deb'
plugin_version=subprocess.run(['dpkg-query','-W','-f=${Version}','saga-vpo'],capture_output=True,text=True).stdout.strip()
if not plugin_version.startswith('2025.6.11'):
    saga_deb = download('https://github.com/DOV-Vlaanderen/saga-watem/releases/download/2025.6.11/'+saga_name, SOFTWARE/saga_name)
    local_deb = Path('/tmp') / saga_name
    shutil.copy2(saga_deb, local_deb)
    run(['apt-get','install','-y',str(local_deb)])
ws = download('https://github.com/watem-sedem/watem-sedem/releases/download/5.0.3/watem_sedem', SOFTWARE/'watem_sedem_5.0.3')
binary = Path('/usr/local/bin/watem_sedem')
shutil.copy2(ws,binary); binary.chmod(0o755)
os.environ['WATEMSEDEM'] = str(binary.parent)
# saga-vpo contains plugins, not the SAGA command-line program.
# Ubuntu's saga package supplies saga_cmd.
saga_cmd = shutil.which('saga_cmd')
if not saga_cmd:
    run(['dpkg','-L','saga'])
    raise RuntimeError('saga_cmd missing after installing saga')
assert shutil.which('watem_sedem'), 'watem_sedem missing from PATH'
if VERBOSE: print('SAGA executable:', saga_cmd)
if VERBOSE: print('WaTEM executable:', shutil.which('watem_sedem'))
plugin_files = subprocess.check_output(['dpkg','-L','saga-vpo'],text=True).splitlines()
if VERBOSE: print('Installed SAGA plugin libraries:', [p for p in plugin_files if p.endswith('.so')])
if VERBOSE: run(['saga_cmd','--version'])
for args in (['saga_cmd','topology'], ['saga_cmd','watem']):
    p = subprocess.run(args,capture_output=True,text=True)
    msg = p.stdout+p.stderr
    if VERBOSE: print('SAGA library check:',args[-1])
    assert 'error: select a library' not in msg.lower() and 'cannot load' not in msg.lower() and 'tools:' in msg.lower(), f'Missing/incompatible SAGA library: {args}\n{msg}'
deps=run(['ldd',str(binary)])
assert 'not found' not in (deps.stdout or ''),deps.stdout
git_env={**os.environ,'GIT_LFS_SKIP_SMUDGE':'1'}
# Keep tags and commit history for setuptools-scm version calculation,
# but defer unrelated historical file blobs. Never download Git LFS objects.
if not REPO.exists():
    run(['git','clone','--filter=blob:none','--no-checkout','https://github.com/watem-sedem/pywatemsedem.git',REPO],env=git_env)
head=subprocess.run(['git','-C',str(REPO),'rev-parse','HEAD'],capture_output=True,text=True).stdout.strip()
if head!=REV or not (REPO/'pyproject.toml').is_file():
    run(['git','-C',REPO,'checkout','--detach',REV],env=git_env)
from importlib.metadata import version, PackageNotFoundError
try:
    installed=version('pywatemsedem')
except PackageNotFoundError:
    installed=''
if not installed.endswith('g'+REV[:9]):
    run([sys.executable,'-m','pip','install',str(REPO)])
import pywatemsedem
done('Software ready: WaTEM/SEDEM 5.0.3 + pinned Python wrapper')

# No Git LFS downloads: all geographical inputs are built for the chosen site.
# Reuse the exact earlier tutorial INI if it is already cached and matches its
# pinned LFS SHA256. Otherwise use an explicit self-contained teaching preset.
inputdata_folder=WORK/'inputs'
inputdata_folder.mkdir(exist_ok=True)
ini_path=inputdata_folder/'userchoices.ini'
expected_ini_sha='75ac6a1519eab406a3dcad491cecceecd66668afcf30c20e68e5c5898c4fd7ee'
cached_ini=DATA/'userchoices.ini'
if cached_ini.is_file() and hashlib.sha256(cached_ini.read_bytes()).hexdigest()==expected_ini_sha:
    shutil.copy2(cached_ini,ini_path)
    SETTINGS_SOURCE='Exact cached upstream Langegracht configuration'
else:
    ini_path.write_text("[Options]\nL model = Desmet1996_Vanoost2003\nS model = Nearing1997\nTC model = VanOost2000\nOnly Routing = 0\nCalculate Tillage Erosion = 0\n\n[Parameters]\nR factor = 1250\nParcel connectivity cropland = 90\nParcel connectivity grasstrips = 30\nParcel connectivity forest = 30\nParcel trapping efficiency cropland = 0\nParcel trapping efficiency pasture = 75\nParcel trapping efficiency forest = 75\nBulk density = 1350\nMax kernel = 50\nMax kernel river = 100\n\n[Extensions]\nCurve Number = 0\nInclude sewers = 0\nCreate ktc map = 0\nCreate ktil map = 0\nEstimate clay content = 0\nInclude tillage direction = 0\nInclude buffers = 0\nInclude ditches = 0\nInclude dams = 0\nOutput per river segment = 0\nAdjusted Slope = 1\nBuffer reduce Area = 0\nForce Routing = 0\nRiver Routing = 0\nManual outlet selection = 0\nConvert output = 0\nCalibrate = 0\nCardinal Routing River = 0\n\n[Parameters extensions]\nktc low = 1\nktc high = 9\nktc limit = 0.1\nLS correction = 1.0\n\n[Output]\nWrite aspect = 1\nWrite LS factor = 1\nWrite upstream area = 1\nWrite slope = 1\nWrite routing table = 1\nWrite routing column/row = 1\nWrite RUSLE = 1\nWrite sediment export = 1\nWrite water erosion = 1\nWrite rainfall excess = 0\nWrite total runoff = 0\nExport .sgrd grids = 0\n")
    SETTINGS_SOURCE='Local teaching preset: Molenbeek 5.0.3 example, pinned wrapper test kTC values, provisional kTC limit 0.1; not verified identical to Langegracht'
if VERBOSE: print('Model settings:',SETTINGS_SOURCE,flush=True)
if VERBOSE: print(ini_path.read_text())
shutil.copy2(ini_path,PACKAGE/'teaching_settings.ini')
(PACKAGE/'settings_provenance.json').write_text(json.dumps(dict(source=SETTINGS_SOURCE,sha256=hashlib.sha256(ini_path.read_bytes()).hexdigest(),upstream_revision=REV),indent=2))

import numpy as np, pandas as pd, geopandas as gpd, rasterio, requests, matplotlib.pyplot as plt
from rasterio.warp import reproject, Resampling, transform_bounds
from rasterio.transform import from_origin, xy
from rasterio.features import shapes, rasterize
from shapely.geometry import shape, Point, box
from pysheds.grid import Grid
from IPython.display import display
CRS='EPSG:31370'
def prepare_terrain(buffer_km,show_maps=True):
    global TERRAIN_BUFFER_KM, xmin, ymin, xmax, ymax, width, height, tr, profile, dem_sources, wc_url, dem, lc, write_raster, sources, session, request_json, fetch_features, search_aoi, river_search, river_meta, BURN_DEPTH, MIN_SLOPE, dem_original, river_cells, saga, to_saga, from_saga, dem_burned, dem_hydro, river_downhill, grid, z, fdir, acc, acc_a, rr, cc, xx, yy, upstream_ha
    bb=transform_bounds('EPSG:4326',CRS,*SEARCH_LONLAT,densify_pts=21)
    xmin,ymin=np.floor(np.array(bb[:2])/RES)*RES; xmax,ymax=np.ceil(np.array(bb[2:])/RES)*RES
    # Expand the terrain window independently of the outlet selection; mosaic all intersecting tiles.
    TERRAIN_BUFFER_KM=float(buffer_km)
    buffer_m=int(np.ceil(TERRAIN_BUFFER_KM*1000/RES))*RES
    xmin-=buffer_m; ymin-=buffer_m; xmax+=buffer_m; ymax+=buffer_m
    print(f'  Terrain buffer: {TERRAIN_BUFFER_KM} km on each side',flush=True)
    width,height=int((xmax-xmin)/RES),int((ymax-ymin)/RES)
    assert width*height<=4000000,'Terrain window exceeds classroom processing limit; use a point or a smaller initial search region in block 1.'
    tr=from_origin(xmin,ymax,RES,RES)
    profile=dict(driver='GTiff',height=height,width=width,count=1,crs=CRS,transform=tr,nodata=-9999,compress='deflate')
    def warp_sources(urls,method):
        dest=np.full((height,width),-9999,dtype='float32')
        for url in urls:
            tmp=np.full_like(dest,-9999)
            with rasterio.open(url) as ds:
                reproject(rasterio.band(ds,1),tmp,dst_transform=tr,dst_crs=CRS,dst_nodata=-9999,resampling=method)
            dest[tmp!=-9999]=tmp[tmp!=-9999]
        assert np.isfinite(dest).all() and (dest!=-9999).all(),'Incomplete raster coverage'
        return dest
    def dem_url(lon,lat):
        tile=f'Copernicus_DSM_COG_10_N{lat:02d}_00_E{lon:03d}_00_DEM'
        return f'https://copernicus-dem-30m.s3.amazonaws.com/{tile}/{tile}.tif'
    import math
    west,south,east,north=transform_bounds(CRS,'EPSG:4326',xmin,ymin,xmax,ymax)
    dem_sources=[str(SPW_DEM)] if SPW_DEM else [dem_url(lon,lat) for lon in range(math.floor(west),math.floor(east)+1) for lat in range(math.floor(south),math.floor(north)+1)]
    wc_url=[f'https://esa-worldcover.s3.eu-central-1.amazonaws.com/v200/2021/map/ESA_WorldCover_10m_2021_v200_N{lat:02d}E{lon:03d}_Map.tif' for lat in range(math.floor(south/3)*3,math.floor(north/3)*3+1,3) for lon in range(math.floor(west/3)*3,math.floor(east/3)*3+1,3)]
    stage('2/5 · Reading elevation and land-cover windows')
    print('  DEM: '+('local SPW raster' if SPW_DEM else 'Copernicus GLO-30'),flush=True)
    if VERBOSE: print(*dem_sources,sep='\n')
    dem=warp_sources(dem_sources,Resampling.bilinear)
    print('  Land cover: ESA WorldCover 2021',flush=True)
    if VERBOSE: print(*wc_url,sep='\n')
    lc=warp_sources(wc_url,Resampling.nearest).astype('int16')
    def write_raster(path,a,dtype='float32',transform=tr):
        p=profile.copy(); p.update(dtype=dtype,height=a.shape[0],width=a.shape[1],transform=transform)
        with rasterio.open(path,'w',**p) as ds: ds.write(a.astype(dtype),1)
    write_raster(WORK/'search_dem.tif',dem)
    done(f'{width} × {height} cells at {RES} m; elevation {dem.min():.0f}–{dem.max():.0f} m')
    stage('3/5 · Retrieving mapped streams from SPW RHW')

    # Retrieve the river network across the FULL buffered DEM region before delineation.
    BASE='https://geoservices.wallonie.be/arcgis/rest/services'
    sources={'parcels':f'{BASE}/AGRICULTURE/SIGEC_PARC_AGRI_{YEAR}/MapServer/4','soils':f'{BASE}/SOL_SOUS_SOL/CNSW/MapServer/2','roads':f'{BASE}/TOPOGRAPHIE/PICC_VDIFF/MapServer/{PICC_IDS["roads"]}','buildings':f'{BASE}/TOPOGRAPHIE/PICC_VDIFF/MapServer/{PICC_IDS["buildings"]}','river':RIVER_URL,'water':f'{BASE}/TOPOGRAPHIE/PICC_VDIFF/MapServer/{PICC_IDS["water"]}'}
    session=requests.Session()
    def request_json(url,params):
        response=session.get(url,params=params,timeout=120); response.raise_for_status()
        j=response.json()
        if 'error' in j: raise RuntimeError(j['error'])
        return j
    def fetch_features(url, aoi=None, out_fields='*', batch_size=100):
        aoi=boundary if aoi is None else aoi
        meta=request_json(url,{'f':'json'})
        assert meta.get('geometryType'),f'Not a feature layer: {url}'
        bounds=aoi.total_bounds
        spatial=dict(where='1=1',geometry=','.join(map(str,bounds)),geometryType='esriGeometryEnvelope',inSR=31370,spatialRel='esriSpatialRelIntersects',returnIdsOnly='true',f='json')
        ids=request_json(url+'/query',spatial).get('objectIds') or []
        features=[]
        for i in range(0,len(ids),batch_size):
            batch=request_json(url+'/query',dict(objectIds=','.join(map(str,ids[i:i+batch_size])),outFields=out_fields,outSR=4326,returnGeometry='true',f='geojson'))
            assert not batch.get('exceededTransferLimit'),'Truncated batch'
            features.extend(batch['features'])
        assert len(features)==len(ids),(len(features),len(ids))
        if not features: return gpd.GeoDataFrame(geometry=[],crs=CRS),meta
        g=gpd.GeoDataFrame.from_features(features,crs=4326).to_crs(CRS)
        g.geometry=g.geometry.make_valid()
        return gpd.clip(g,aoi,keep_geom_type=True),meta

    search_aoi=gpd.GeoDataFrame(geometry=[box(xmin,ymin,xmax,ymax)],crs=CRS)
    river_search,river_meta=fetch_features(sources['river'],search_aoi,out_fields='OBJECTID',batch_size=200)
    assert river_meta['geometryType']=='esriGeometryPolyline' and len(river_search)>0
    river_search=river_search.explode(index_parts=False).reset_index(drop=True)
    river_search=river_search[river_search.geometry.geom_type=='LineString'].copy()
    assert len(river_search)>0
    river_search.to_file(WORK/'river_search.gpkg',driver='GPKG')
    done(f'{len(river_search)} stream lines retrieved')
    stage('4/5 · Conditioning the DEM and checking drainage')
    # SAGA native grids via GDAL conversion; stream background must be NoData, not zero.
    # Method 0 avoids inventing an authoritative direction from an unoriented vector network.
    # A post-fill D8 check then requires mapped river connectivity within selected basin.
    BURN_DEPTH=1.0
    MIN_SLOPE=0.01
    dem_original=dem.copy()
    river_cells=rasterize([(g,1) for g in river_search.geometry],out_shape=dem.shape,transform=tr,fill=0,all_touched=True,dtype='uint8').astype(bool)
    write_raster(WORK/'stream_mask.tif',np.where(river_cells,1,-9999),'int16')
    def saga(*args):
        cmd=['saga_cmd',*map(str,args)]
        proc=subprocess.run(cmd,capture_output=True,text=True,errors='replace')
        if VERBOSE: print(' '.join(cmd),proc.returncode,flush=True)
        if proc.returncode: print(proc.stdout[-4000:],proc.stderr[-4000:]); proc.check_returncode()
    def to_saga(src,dst):
        # Import with SAGA's GDAL tool (preserves NoData and grid geometry).
        saga('io_gdal',0,'-FILES',src,'-GRIDS',dst)
        assert Path(dst).exists(),dst
    def from_saga(src,dst):
        saga('io_gdal',2,'-GRIDS',src,'-FILE',dst)
        with rasterio.open(dst) as ds:
            assert ds.shape==dem.shape and ds.transform.almost_equals(tr), (ds.shape,ds.transform,tr)
            return ds.read(1).astype('float32')
    # Remove earlier SAGA outputs so a larger terrain grid cannot reuse stale files.
    for stem in ('original','streams','burned','hydro','hydro_fdir','hydro_wshed'):
        for ext in ('.sgrd','.sdat','.mgrd','.prj','.sdat.aux.xml'):
            (WORK/(stem+ext)).unlink(missing_ok=True)
    for name in ('search_dem_burned.tif','search_dem_hydro_export.tif'):
        (WORK/name).unlink(missing_ok=True)
    to_saga(WORK/'search_dem.tif',WORK/'original.sgrd')
    to_saga(WORK/'stream_mask.tif',WORK/'streams.sgrd')
    saga('ta_preprocessor',6,'-DEM',WORK/'original.sgrd','-STREAM',WORK/'streams.sgrd','-BURN',WORK/'burned.sgrd','-METHOD',0,'-EPSILON',BURN_DEPTH)
    saga('ta_preprocessor',4,'-ELEV',WORK/'burned.sgrd','-FILLED',WORK/'hydro.sgrd','-FDIR',WORK/'hydro_fdir.sgrd','-WSHED',WORK/'hydro_wshed.sgrd','-MINSLOPE',MIN_SLOPE)
    dem_burned=from_saga(WORK/'burned.sgrd',WORK/'search_dem_burned.tif')
    dem_hydro=from_saga(WORK/'hydro.sgrd',WORK/'search_dem_hydro_export.tif')
    assert np.isfinite(dem_hydro).all() and (dem_hydro!=-9999).all()
    write_raster(WORK/'search_dem_hydro.tif',dem_hydro)
    # Independent steepest D8 slope check, using pysheds default D8 ordering.
    # Interior cells only: boundary cells may legitimately drain outside search domain.
    dirs=[(-1,0),(-1,1),(0,1),(1,1),(1,0),(1,-1),(0,-1),(-1,-1)]
    best=np.full(dem.shape,-np.inf); river_downhill=np.zeros(dem.shape,dtype=bool)
    for dr,dc in dirs:
        nbr=np.roll(dem_hydro,(-dr,-dc),axis=(0,1))
        nriv=np.roll(river_cells,(-dr,-dc),axis=(0,1))
        slope=(dem_hydro-nbr)/(RES*np.hypot(dr,dc))
        better=slope>best
        river_downhill[better]=nriv[better] & (slope[better]>0)
        best[better]=slope[better]
    interior=np.ones(dem.shape,dtype=bool); interior[[0,-1],:]=False; interior[:,[0,-1]]=False
    pits=interior & (best<=0)
    assert not pits.any(),f'{pits.sum()} interior cells lack a downhill neighbour'
    bad_river=river_cells & interior & ~river_downhill
    if VERBOSE: print('Interior pits/flats:',int(pits.sum()),'Mapped stream cells without a mapped downhill successor:',int(bad_river.sum()))
    delta=dem_hydro-dem_original
    done(f'No interior pits; elevation change min/median/max: {delta.min():.1f} / {np.median(delta):.1f} / {delta.max():.1f} m')
    fig,axes=plt.subplots(1,3,figsize=(16,6),constrained_layout=True)
    for ax,a,title in zip(axes,[dem_original,dem_hydro,delta],['Original DEM','Hydrologically conditioned DEM','Elevation modification (m)']):
        im=ax.imshow(a,cmap='terrain' if title!='Elevation modification (m)' else 'RdBu_r')
        ax.contour(river_cells,levels=[.5],colors='blue',linewidths=.3)
        ax.set_title(title); fig.colorbar(im,ax=ax,shrink=.65)
    fig.savefig(WORK/'hydro_conditioning.png',dpi=180)
    if show_maps: plt.show()
    plt.close(fig)


    # Compatibility with NumPy >= 2.4: pysheds still uses the removed np.in1d.
    # Preserve the old one-dimensional return shape and signature.
    if not hasattr(np, 'in1d'):
        def _in1d(ar1, ar2, assume_unique=False, invert=False):
            return np.isin(np.asarray(ar1).ravel(), np.asarray(ar2).ravel(),
                           assume_unique=assume_unique, invert=invert)
        np.in1d = _in1d
        if VERBOSE: print('Applied pysheds / NumPy in1d compatibility shim.')
    stage('5/5 · Calculating D8 flow directions and accumulation (first run compiles numerical kernels)')
    grid=Grid.from_raster(str(WORK/'search_dem_hydro.tif'))
    z=grid.read_raster(str(WORK/'search_dem_hydro.tif'))
    fdir=grid.flowdir(z)
    acc=grid.accumulation(fdir)
    acc_a=np.asarray(acc)
    # candidate outlet within original search bbox (not the surrounding buffer)
    rr,cc=np.indices(dem.shape)
    xx=xmin+(cc+.5)*RES; yy=ymax-(rr+.5)*RES
    upstream_ha=acc_a*RES**2/10000
    done('Flow directions and contributing area ready')

prepare_terrain(TERRAIN_BUFFER_STEPS[0])
ORIGINAL_OUTLET=OUTLET_POINT
confirmed_outlet=None
print(f'Block 2 completed in {(perf_counter()-_block_start)/60:.1f} min. Confirm the outlet in block 3.',flush=True)


## 3. Select the exact model outlet
Blue = official mapped streams; yellow = D8 contributing-area network. Click a yellow line, optionally drag the marker, then explicitly confirm. The display threshold is not the model river threshold.

In [ ]:
#@title 3 — Confirm exact hydrological outlet
# Interactive confirmation on the DEM-derived drainage network; no terrain download.
from google.colab import output
from IPython.display import HTML, JSON
import uuid
from pyproj import Transformer
from shapely.geometry import LineString, mapping
from IPython.display import display
to_ll=Transformer.from_crs(CRS,4326,always_xy=True)
to_xy=Transformer.from_crs(4326,CRS,always_xy=True)
confirmed_outlet=None
outlet_state={'candidate':None,'confirmed':False}
if ORIGINAL_OUTLET is not None:
    cx,cy=to_xy.transform(ORIGINAL_OUTLET.x,ORIGINAL_OUTLET.y)
else:
    cx,cy=to_xy.transform(AOI.centroid.x,AOI.centroid.y)
# Render only high-accumulation D8 cells as short connected vector segments.
# No full raster or opaque low-accumulation background is transferred to the browser.
eligible=(upstream_ha>=MIN_CATCHMENT_HA)
eligible[[0,-1],:]=False; eligible[:,[0,-1]]=False
offsets={64:(-1,0),128:(-1,1),1:(0,1),2:(1,1),4:(1,0),8:(1,-1),16:(0,-1),32:(-1,-1)}
fd=np.asarray(fdir)
# Calculate exact area and edge provenance on the identical D8 graph used in block 4.
# Any cell downstream of a terrain-edge cell has potentially incomplete coverage.
from numba import njit
@njit
def outlet_graph(directions,elevation):
    h,w=directions.shape
    dest=np.full(h*w,-1,np.int64)
    count=np.ones(h*w,np.int64)
    edge=np.zeros(h*w,np.uint8)
    codes=(64,128,1,2,4,8,16,32)
    drs=(-1,-1,0,1,1,1,0,-1)
    dcs=(0,1,1,1,0,-1,-1,-1)
    for r in range(h):
        for c in range(w):
            i=r*w+c
            if r==0 or c==0 or r==h-1 or c==w-1: edge[i]=1
            for k in range(8):
                if directions[r,c]==codes[k]:
                    a,b=r+drs[k],c+dcs[k]
                    if 0<=a<h and 0<=b<w:
                        if elevation[a,b]>=elevation[r,c]:
                            raise ValueError('Non-downhill D8 link')
                        dest[i]=a*w+b
                    break
    order=np.argsort(elevation.ravel())
    for q in range(order.size-1,-1,-1):
        i=order[q]; j=dest[i]
        if j>=0:
            count[j]+=count[i]
            edge[j]=max(edge[j],edge[i])
    return count.reshape((h,w)),edge.reshape((h,w))
map_acc,map_edge=outlet_graph(fd,np.asarray(dem_hydro))
map_area=map_acc*RES**2/10000
eligible=(map_area>=MIN_CATCHMENT_HA)
eligible[[0,-1],:]=False; eligible[:,[0,-1]]=False
print('Outlet map checked for upstream terrain-edge connectivity.')
def drainage_geojson(threshold):
    # Vectorized conversion; group short segments into one MultiLineString.
    rr0,cc0=np.where(upstream_ha>=threshold)
    code=fd[rr0,cc0]
    dr=np.zeros(len(rr0),dtype=int); dc=dr.copy(); valid=np.zeros(len(rr0),dtype=bool)
    for k,(dy,dx) in offsets.items():
        hit=code==k; dr[hit]=dy; dc[hit]=dx; valid|=hit
    rt=rr0+dr; ct=cc0+dc
    valid &= (rt>=0)&(rt<fd.shape[0])&(ct>=0)&(ct<fd.shape[1])
    rr0,cc0,rt,ct=[v[valid] for v in (rr0,cc0,rt,ct)]
    x1,y1=to_ll.transform(xx[rr0,cc0],yy[rr0,cc0]); x2,y2=to_ll.transform(xx[rt,ct],yy[rt,ct])
    coordinates=np.round(np.stack([np.column_stack([x1,y1]),np.column_stack([x2,y2])],axis=1),6).tolist()
    return dict(type='FeatureCollection',features=[dict(type='Feature',geometry=dict(type='MultiLineString',coordinates=coordinates),properties={})] if coordinates else []),len(coordinates)

# Use browser-native Leaflet, not the Colab ipyleaflet widget bridge.
# All geometries are embedded in the same HTML document before map initialization.
print('Preparing official RHW and D8 drainage data for the browser…',flush=True)
clon,clat=to_ll.transform(cx,cy)
rhw_data=json.loads(river_search[['geometry']].to_crs(4326).to_json())
# Convert all D8 segments once at the lowest display threshold. Each has its area.
ri,ci=np.where((map_area>=10)&np.isin(fd,list(offsets)))
dr=np.zeros(len(ri),dtype=int); dc=dr.copy()
for code,(dy,dx) in offsets.items():
    hit=fd[ri,ci]==code; dr[hit]=dy; dc[hit]=dx
rt,ct=ri+dr,ci+dc
valid=(rt>=0)&(rt<fd.shape[0])&(ct>=0)&(ct<fd.shape[1])
ri,ci,rt,ct=[v[valid] for v in (ri,ci,rt,ct)]
l1,b1=to_ll.transform(xx[ri,ci],yy[ri,ci]); l2,b2=to_ll.transform(xx[rt,ct],yy[rt,ct])
segments=np.round(np.column_stack([b1,l1,b2,l2,map_area[ri,ci],map_edge[ri,ci]]),6).tolist()
er,ec=np.where(eligible & np.isin(fd,list(offsets)))
el,eb=to_ll.transform(xx[er,ec],yy[er,ec])
points=np.column_stack([eb,el,er,ec,map_area[er,ec],river_cells[er,ec].astype(int),map_edge[er,ec]]).tolist()
assert len(points),'No eligible D8 outlet cells in prepared terrain'
token='watem_'+uuid.uuid4().hex
callback=token+'_confirm'
def confirm_browser_outlet(action,ri=None,ci=None):
    global confirmed_outlet
    if action=='clear':
        confirmed_outlet=None; outlet_state.update(candidate=None,confirmed=False)
        return JSON({'ok':True})
    ri,ci=int(ri),int(ci)
    assert 0<=ri<eligible.shape[0] and 0<=ci<eligible.shape[1] and eligible[ri,ci]
    assert int(fd[ri,ci]) in offsets
    assert not map_edge[ri,ci],'Incomplete upstream terrain coverage; select a yellow or orange channel.'
    assert map_area[ri,ci]<=MAX_CATCHMENT_HA,'Outlet exceeds classroom size limit; choose a tributary upstream'
    confirmed_outlet=(ri,ci); outlet_state.update(candidate=confirmed_outlet,confirmed=True)
    return JSON({'ok':True,'area_ha':float(map_area[ri,ci])})
output.register_callback(callback,confirm_browser_outlet)
payload=json.dumps(dict(center=[clat,clon],rhw=rhw_data,segments=segments,points=points,callback=callback,minimum=MIN_CATCHMENT_HA,maximum=MAX_CATCHMENT_HA,recommended=RECOMMENDED_CATCHMENT_HA,original=ORIGINAL_OUTLET is not None),separators=(',',':'),allow_nan=False).replace('</','<\\/')
html=r"""
<div id="TOKEN" style="font:14px system-ui;color:#222;background:white;padding:8px">
<div style="font-size:18px;font-weight:600;margin-bottom:8px">Confirm hydrological outlet</div>
<div><b>Step 1:</b> Click a D8 channel to place the outlet, or drag the marker. Yellow = recommended; orange = larger; red = too large; grey = incomplete upstream terrain coverage. Blue = mapped RHW.</div>
<div style="display:flex;gap:8px;flex-wrap:wrap;margin:10px 0">
<select class="base"><option>Satellite</option><option>Topographic</option><option>Street</option></select>
<label>Show upstream area ≥ <select class="threshold"><option>10</option><option selected>25</option><option>50</option><option>100</option><option>250</option><option>500</option></select> ha</label>
<button class="redraw">Redraw rivers</button>
</div>
<div class="status" role="status" style="padding:10px;background:#eaf3ff;margin:6px 0">Loading Leaflet map library…</div>
<div class="map" style="height:580px;width:100%;background:#eee"></div>
<div class="info" style="padding:10px">Step 1: Select an outlet manually on the map. No outlet has been selected.</div>
<button class="confirm" disabled style="padding:9px 18px;background:#18783b;color:white;border:0;border-radius:5px">Step 2: Confirm selected outlet</button>
</div>
<script>
(async function(){
const root=document.getElementById('TOKEN'), q=s=>root.querySelector(s), D=PAYLOAD;
const status=q('.status'), info=q('.info'), btn=q('.confirm');
function say(t){status.textContent=t;}
function loadScript(url){return new Promise((resolve,reject)=>{const e=document.createElement('script');e.src=url;e.onload=resolve;e.onerror=reject;document.head.appendChild(e);});}
try{
 const css=document.createElement('link');css.rel='stylesheet';css.href='https://unpkg.com/leaflet@1.9.4/dist/leaflet.css';document.head.appendChild(css);
 if(!window.L)await loadScript('https://unpkg.com/leaflet@1.9.4/dist/leaflet.js');
 say('Initializing map and both river overlays…');
 const map=L.map(q('.map'),{preferCanvas:false,maxZoom:20}).setView(D.center,15);
 const esri='https://server.arcgisonline.com/ArcGIS/rest/services/';
 const basemaps={'Satellite':'World_Imagery','Topographic':'World_Topo_Map','Street':'World_Street_Map'};
 let base, tileState='loading';
 function setBase(){if(base)map.removeLayer(base);base=L.tileLayer(esri+basemaps[q('.base').value]+'/MapServer/tile/{z}/{y}/{x}',{maxNativeZoom:19,maxZoom:20,attribution:'Esri and contributing providers'});base.on('loading',()=>{tileState='loading';report();});base.on('load',()=>{tileState='ready';report();});base.on('tileerror',()=>{tileState='tile error';report();});base.addTo(map);}
 let rhw,drain,n=0,overlayReady=false;
 function report(){say(overlayReady?'Rivers drawn: '+D.rhw.features.length+' RHW features; '+n+' D8 segments. Basemap: '+tileState+'.':'Preparing river overlays…');}
 map.createPane('rhw');map.getPane('rhw').style.zIndex=410;map.getPane('rhw').style.pointerEvents='none';
 map.createPane('d8');map.getPane('d8').style.zIndex=420;map.getPane('d8').style.pointerEvents='none';
 const rhwRenderer=L.svg({pane:'rhw',padding:.5});rhwRenderer.addTo(map);
 const d8Renderer=L.svg({pane:'d8',padding:.5});d8Renderer.addTo(map);
 function draw(){overlayReady=false;say('Drawing RHW and D8 layers…');if(rhw)map.removeLayer(rhw);if(drain)map.removeLayer(drain);
 rhw=L.geoJSON(D.rhw,{pane:'rhw',renderer:rhwRenderer,interactive:false,style:{color:'#00bfff',weight:3,opacity:1}}).addTo(map);
 const selected=D.segments.filter(s=>s[4]>=Number(q('.threshold').value));n=selected.length;
 drain=L.layerGroup();for(const [color,filter] of [['#ffdf00',s=>!s[5]&&s[4]<=D.recommended],['#ff9b27',s=>!s[5]&&s[4]>D.recommended&&s[4]<=D.maximum],['#9ca3af',s=>s[5]&&s[4]<=D.maximum],['#ef4444',s=>s[4]>D.maximum]]){const paths=selected.filter(filter).map(s=>[[s[0],s[1]],[s[2],s[3]]]);if(paths.length)L.polyline(paths,{pane:'d8',renderer:d8Renderer,color,weight:4,opacity:1,interactive:false}).addTo(drain);}drain.addTo(map);
 map.invalidateSize();overlayReady=true;report();}
 setBase();await new Promise(requestAnimationFrame);draw();q('.base').onchange=setBase;q('.threshold').onchange=draw;q('.redraw').onclick=draw;
 L.control.scale().addTo(map);
 if(D.original)L.circleMarker(D.center,{radius:7,color:'#fff',fillColor:'#e53935',fillOpacity:1,interactive:false}).addTo(map);
 const icon=L.divIcon({className:'',html:'<div style="width:18px;height:18px;border-radius:50%;background:#16a34a;border:3px solid white;box-shadow:0 0 3px black"></div>',iconSize:[24,24],iconAnchor:[12,12]});
 const marker=L.marker(D.center,{draggable:true,icon,title:'Drag outlet'});
 let chosen=null, revision=0, pending=Promise.resolve();
 function kernel(action,p){return google.colab.kernel.invokeFunction(D.callback,action==='clear'?['clear']:['confirm',p[2],p[3]],{});}
 function invalidate(){revision++;btn.disabled=true;btn.textContent='Step 2: Confirm selected outlet';info.style.background='';chosen=null;pending=pending.catch(()=>{}).then(()=>kernel('clear')).catch(e=>{info.textContent='Could not clear Python confirmation: '+e;});}
 function pick(ll,radius){invalidate();let best=null,dist=Infinity;
 for(const p of D.points){const d=map.distance(ll,[p[0],p[1]]);if(d<dist){dist=d;best=p;}}
 if(!best||dist>radius){info.textContent='No eligible cell within '+radius+' m. Select a nearby channel. Supported catchment area: '+D.minimum+'–'+D.maximum+' ha.';return;}
 marker.setLatLng([best[0],best[1]]);if(!map.hasLayer(marker))marker.addTo(map);if(best[4]>D.maximum){info.textContent='Too large: at least '+best[4].toFixed(0)+' ha. Select an upstream tributary (limit '+D.maximum+' ha).';info.style.background='#fee2e2';return;}if(best[6]){info.textContent='Incomplete upstream terrain coverage: at least '+best[4].toFixed(0)+' ha. Choose a yellow or orange tributary.';info.style.background='#e5e7eb';return;}chosen=best;info.textContent='Checked upstream area: '+best[4].toFixed(1)+' ha | '+(best[4]>D.recommended?'Large basin: processing may be slower. ':'Recommended size. ')+(best[5]?'on RHW':'off RHW; D8 channel used');info.style.background=best[4]>D.recommended?'#fff1db':'';btn.disabled=false;}
 marker.on('dragstart',invalidate);marker.on('dragend',()=>pick(marker.getLatLng(),40));map.on('click',e=>pick(e.latlng,40));
 btn.onclick=async()=>{if(!chosen)return;const p=chosen,rev=revision;btn.disabled=true;info.textContent='Sending confirmed outlet to Python…';try{await pending;if(rev!==revision)return;pending=kernel('confirm',p);await pending;if(rev===revision){info.textContent='✓ OUTLET CONFIRMED — '+p[4].toFixed(1)+' ha. Step 3: Run the next notebook cell to delineate the watershed.';info.style.background='#dcfce7';btn.textContent='✓ Outlet confirmed';}}catch(e){info.textContent='Confirmation failed: '+e;btn.disabled=false;}};
 // Repaint on actual container resize, not on a basemap switch.
 if(window.ResizeObserver)new ResizeObserver(()=>map.invalidateSize()).observe(q('.map'));
 requestAnimationFrame(()=>{map.invalidateSize();draw();});info.textContent='Step 1: Click a yellow or orange drainage line to select the outlet. Nothing is selected yet.';
}catch(e){say('Map initialization failed: '+e+'. Check access to unpkg.com and rerun this cell.');console.error(e);}
})();
</script>
"""
display(HTML(html.replace('TOKEN',token).replace('PAYLOAD',payload)))


## 4. Delineate and inspect the watershed
The original D8 watershed raster defines the active domain. Check its area, boundary and agricultural fraction. If unsuitable, return to block 3, move and confirm the outlet and rerun block 4. Only continue if the watershed is acceptable.

In [ ]:
#@title 4 — Delineate watershed
assert outlet_state['confirmed'] and confirmed_outlet is not None,'Confirm the outlet on the map first.'
# Independently delineate and accumulate from the SAME immutable D8 graph.
# pysheds catchment/accumulation may treat raster rims differently; use an explicit
# reverse traversal and a topological accumulation with identical edge semantics.
D8_OFFSETS={64:(-1,0),128:(-1,1),1:(0,1),2:(1,1),4:(1,0),8:(1,-1),16:(0,-1),32:(-1,-1)}
from collections import deque
from rasterio.transform import rowcol
from pyproj import Transformer
# Preserve the exact confirmed cell centre in projected metres across larger grids.
outlet_x,outlet_y=xy(tr,*map(int,confirmed_outlet))
def reverse_basin(fd,row,col):
    h,w=fd.shape
    basin=np.zeros((h,w),dtype=bool)
    basin[row,col]=True
    queue=deque([(row,col)])
    while queue:
        r,c=queue.popleft()
        for code,(dr,dc) in D8_OFFSETS.items():
            a,b=r-dr,c-dc
            if 0<=a<h and 0<=b<w and not basin[a,b] and fd[a,b]==code:
                basin[a,b]=True
                queue.append((a,b))
    return basin
while True:
    fd=np.asarray(fdir).copy()
    h,w=fd.shape
    row,col=map(int,rowcol(tr,outlet_x,outlet_y))
    assert 0<row<h-1 and 0<col<w-1
    assert np.allclose(xy(tr,row,col),[outlet_x,outlet_y],atol=1e-5,rtol=0),'Outlet grid alignment changed'
    assert int(fd[row,col]) in D8_OFFSETS,'Confirmed outlet no longer has a valid D8 successor; reselect in block 3.'
    mask=reverse_basin(fd,row,col)
    area=float(mask.sum()*RES**2/10000)
    edge_counts={'north':int(mask[0,:].sum()),'south':int(mask[-1,:].sum()),'west':int(mask[:,0].sum()),'east':int(mask[:,-1].sum())}
    if area>MAX_CATCHMENT_HA:
        raise AssertionError(f'Upstream area is at least {area:,.0f} ha, above the classroom limit of {MAX_CATCHMENT_HA:,} ha. Select an upstream tributary in block 3.')
    if not any(edge_counts.values()):
        break
    raise AssertionError('Selected watershed reaches the terrain boundary. Return to block 3 and select a complete yellow or orange tributary, or choose a new approximate location in block 1. Incomplete basins are never simulated.')
assert MIN_CATCHMENT_HA<=area<=MAX_CATCHMENT_HA, f'Final area {area:.2f} ha outside supported range; select another outlet.'
confirmed_outlet=(row,col)
outlet_state.update(candidate=confirmed_outlet,confirmed=True)
if area>RECOMMENDED_CATCHMENT_HA:
    print(f'Large catchment: {area:,.0f} ha. Input downloads and simulations may take longer.')
# Independent topological count, on the full grid, with each cell contributing one.
# A strictly downhill graph is acyclic. Process cells in descending elevation order.
dest=np.full((h,w),-1,dtype=np.int64)
rr_all,cc_all=np.indices((h,w))
for code,(dr,dc) in D8_OFFSETS.items():
    rt,ct=rr_all+dr,cc_all+dc
    valid=(fd==code)&(rt>=0)&(rt<h)&(ct>=0)&(ct<w)
    dest[valid]=rt[valid]*w+ct[valid]
flatdest=dest.ravel()
src=np.flatnonzero(flatdest>=0)
elev=np.asarray(dem_hydro).ravel()
assert np.all(elev[flatdest[src]]<elev[src]),'D8 graph contains non-downhill links'
counts=np.ones(h*w,dtype=np.int64)
for ix in np.argsort(elev,kind='stable')[::-1]:
    j=flatdest[ix]
    if j>=0: counts[j]+=counts[ix]
checked_acc=counts.reshape(h,w)
# Validate membership against pysheds too; differences are reported, not hidden.
pysheds_count=int(acc_a[row,col])
pysheds_mask=np.asarray(grid.catchment(x=col,y=row,fdir=fdir.copy(),xytype='index'),dtype=bool)
print(f'D8 QA: original accumulation {pysheds_count:,}; original delineation {pysheds_mask.sum():,}; independent graph {mask.sum():,}; independent accumulation {checked_acc[row,col]:,} cells')
print('Delineation membership difference:',int(np.count_nonzero(pysheds_mask!=mask)),'cells')
assert int(checked_acc[row,col])==int(mask.sum()),'Independent accumulation and reverse traversal disagree'
# Use the independently validated accumulation consistently for all later channel thresholds.
acc_a=checked_acc
upstream_ha=acc_a*RES**2/10000
bad=mask & river_cells & ~river_downhill
bad[row,col]=False
print(f'Confirmed outlet: {area:.2f} ha; mapped RHW cell: {bool(river_cells[row,col])}')
print('Mapped stream alignment:',int(bad.sum()),'of',int((mask & river_cells).sum()),'cells lack a mapped D8 successor (diagnostic only).')
print('D8 closure PASS; burned river cells:',int((mask & river_cells).sum()))
misaligned=mask & river_cells & ~river_downhill
misaligned[row,col]=False
fig,ax=plt.subplots(figsize=(9,7)); ax.imshow(np.ma.masked_where(~mask,dem_hydro),cmap='terrain'); ax.contour(mask,levels=[.5],colors='black',linewidths=.7)
r,c=np.where(mask & river_cells); ax.scatter(c,r,s=3,c='blue',label='Mapped stream cells')
r,c=np.where(misaligned); ax.scatter(c,r,s=9,c='orange',label='Mapped cells with non-stream D8 successor')
ax.scatter(col,row,s=40,c='red',label='Outlet'); ax.legend(); ax.set_title('River alignment diagnostic (not D8 closure)'); plt.show()
outxy=xy(tr,row,col)
boundary=gpd.GeoDataFrame(geometry=[shape(g) for g,v in shapes(mask.astype('uint8'),mask=mask,transform=tr) if v==1],crs=CRS).dissolve()
outlet=gpd.GeoDataFrame(geometry=[Point(*outxy)],crs=CRS)
print('Chosen area ha:',mask.sum()*RES**2/10000,'outlet lon/lat:',outlet.to_crs(4326).geometry.iloc[0])
boundary.to_file(PACKAGE/'catchment.shp'); outlet.to_file(PACKAGE/'outlet.gpkg',driver='GPKG')
if OUTLET_POINT is not None:
    gpd.GeoDataFrame(geometry=[OUTLET_POINT],crs=4326).to_crs(CRS).to_file(PACKAGE/'requested_outlet.gpkg',driver='GPKG')
fig,ax=plt.subplots(figsize=(9,8)); boundary.plot(ax=ax,facecolor='#e6e5d8',edgecolor='black'); gpd.clip(river_search,boundary).plot(ax=ax,color='blue',linewidth=2); outlet.plot(ax=ax,color='red',markersize=40); ax.set_title('Selected DEM-derived basin | mapped streams and outlet'); plt.show()
print(f'Agriculture/grass: {100*np.mean(np.isin(lc[mask],[30,40])):.1f}% | D8 area: {mask.sum()*RES**2/10000:.2f} ha')


## 5. Download catchment data and convert to model inputs
Tick ACCEPT only after inspecting the watershed. The following layers are clipped to the basin and exported for GIS inspection.

| Downloaded dataset | Role |
|---|---|
| SPW SIGEC agricultural parcels (2025) | Crop labels → provisional crop C and CN, parcel IDs |
| SPW CNSW soil map | Soil interpretation only; K and HSG are not yet derived from it |
| SPW PICC roads, buildings, standing water | Infrastructure and water classes in PRC |
| SPW RHW streams | Reference and terrain conditioning; NOT the final model river raster |
| ESA WorldCover | Background forest, grass, cropland, water and built-up classes |

Final inputs: aligned conditioned DTM; exact D8 mask; PRC (positive parcel IDs and negative land-use codes, zero outside); C-factor raster with forest 0.001 and grass 0.01; provisional uniform K=30 and HSG B; CN raster; P and transport-capacity settings from tutorial; model river cells defined by contributing area ≥ MODEL_RIVER_HA. Native Idrisi exports are checked against the original D8 domain. The preview C map is not the final native C raster.

**CN extension:** the native implementation includes intensity correction `(I10/10)^alpha`, antecedent rainfall `AR5*beta` and negative rainfall excess representing potential reinfiltration. Defaults leave the first two corrections off; the negative deficit mechanism remains active. The cover/roughness CN equation is not used by the current lookup raster. Nonzero coefficients are experimental pending verification against the original thesis; AR5 is supplied in mm, not downloaded.

Event channel routing is enabled by default via ROUTE_CHANNEL_RUNOFF. This is explicit cell-to-cell D8 routing with the confirmed outlet, not native segment topology. Disable it to reproduce the earlier raster-only baseline.

### Parameter sources and assumptions

**C-factor:** the Walloon NVE manual reproduces the GISER crop lookup (Bielders et al. 2011, as cited in the manual). The existing crop values match its table: rapeseed 0.38, beet 0.41, cereals 0.39 (unweighted mean), maize 0.61, potatoes 0.64, flax/hemp 0.68. Grass 0.01, forest 0.001, other land 0.37 and water/infrastructure 0 are also supported by its other-cover table. The generic crop fallback is our application of the other-land value, not a crop-specific estimate. The values are annual estimates; using them unchanged for individual storms is a teaching simplification.

**Curve Number:** USDA SCS (1986), *Urban Hydrology for Small Watersheds*, TR-55, tables 2-2a–c, reproduced by USACE HEC-HMS. HSG B and average antecedent runoff condition: CN 81 for straight-row crops in poor hydrologic condition; 76 for small grain in poor condition; 61 for pasture in good condition; 55 for woods in good condition; 98 for impervious surfaces. Applying row-crop CN 81 to every non-cereal crop and grassland CN 61 to grass strips is an analogy, not a crop-specific calibration. Open water and river CN 100 are modelling assumptions. HSG B is uniform and provisional; the soil map is not yet used to estimate HSG. The modified CN intensity and antecedent-rainfall terms are disabled by default.

**References**
- [Walloon NVE manual — Protection contre la perte en sol et l’érosion hydrique diffuse](https://natuurwaardeverkenner.marvintest.vito.be/docs/manual_FR/regulerende_diensten/Reguleren_stromen/erosie/) (GISER crop table and other-cover table).
- [USDA SCS TR-55 curve-number tables, reproduced by USACE HEC-HMS](https://www.hec.usace.army.mil/confluence/hmsdocs/hmstrm/cn-tables).
- [Panagos et al. (2015), Estimating the soil erosion cover-management factor at the European scale, Land Use Policy 48, 38–50](https://esdac.jrc.ec.europa.eu/content/cover-management-factor-c-factor-eu) — alternative European approach accounting for crop composition and management.
- [WaTEM/SEDEM model input documentation](https://watem-sedem.github.io/watem-sedem/input.html) — parameter definitions and time-scale consistency.

K and kTC remain provisional, not locally calibrated. The C and CN values are unchanged in this revision.


In [ ]:
#@title 5 — Accept basin, download inputs, build and verify native model
ACCEPT=False #@param {type:'boolean'}
assert ACCEPT,'Inspect the watershed and tick ACCEPT before continuing'

# Provisional hydrology settings. Defaults reproduce the successful baseline.
CN_ALPHA=0.0 #@param {type:"number"}
CN_BETA=0.0 #@param {type:"number"}
CN_AR5_MM=0.0 #@param {type:"number"}
ROUTE_CHANNEL_RUNOFF=True #@param {type:'boolean'}
assert np.isfinite([CN_ALPHA,CN_BETA,CN_AR5_MM]).all() and CN_AR5_MM>=0
print('CN corrections: alpha=',CN_ALPHA,'beta=',CN_BETA,'AR5 (mm)=',CN_AR5_MM)
if CN_ALPHA!=0 or CN_BETA!=0:
    print('EXPERIMENTAL: coefficient units/calibration not verified against the original thesis. Do not interpret as calibrated runoff.')


vectors={}; metadata={}
for name,url in sources.items():
    print('Retrieving',name,flush=True)
    if VERBOSE: print(url)
    if name=='river':
        vectors[name]=gpd.clip(river_search,boundary,keep_geom_type=True)
        metadata[name]=river_meta
    else:
        vectors[name],metadata[name]=fetch_features(url)
    expected={'parcels':'esriGeometryPolygon','soils':'esriGeometryPolygon','roads':'esriGeometryPolyline','buildings':'esriGeometryPolygon','river':'esriGeometryPolyline','water':'esriGeometryPolygon'}[name]
    assert metadata[name]['geometryType']==expected,(name,metadata[name]['geometryType'])
    if name=='water' and 'NATUR_DESC' in vectors[name]:
        vectors[name]=vectors[name][vectors[name].NATUR_DESC!='Piscine'].copy()
    if VERBOSE: print(name,len(vectors[name]),list(vectors[name].columns))
    if len(vectors[name]): vectors[name].to_file(PACKAGE/f'{name}_source.gpkg',driver='GPKG')
    (PACKAGE/f'{name}_schema.json').write_text(json.dumps(metadata[name],indent=2))
assert len(vectors['parcels']) and len(vectors['soils'])
if VERBOSE:
    display(vectors['parcels'].drop(columns='geometry').head())
    display(vectors['soils'].drop(columns='geometry').head())

p=vectors['parcels'].copy()
fields=metadata['parcels'].get('fields',[])
crop_fields=[f['name'] for f in fields if any(k in (f.get('name','')+' '+f.get('alias','')).lower() for k in ['culture','crop'])]
if VERBOSE: print('Candidate crop fields:',crop_fields)
# User can set CROP_FIELD explicitly after viewing source schema.
CROP_FIELD='CULT_NOM'
assert CROP_FIELD in p.columns,'Identify official crop attribute in schema'
# Decode coded value domain where provided
field=next(f for f in fields if f['name']==CROP_FIELD)
domain={str(x['code']):x['name'] for x in (field.get('domain') or {}).get('codedValues',[])}
p['crop_label']=p[CROP_FIELD].astype(str).map(lambda x:domain.get(x,x))
def crop_params(label):
    import re, unicodedata
    x=unicodedata.normalize('NFKD',label.lower())
    x=''.join(ch for ch in x if not unicodedata.combining(ch))
    if any(re.search(r'\b'+re.escape(k)+r'\w*\b',x) for k in ['enherbe','herbace','luzerne','trefle','miscanthus']):
        return .01,61,'grass'  # provisional perennial/vegetated-cover analogy
    if re.search(r'\b(chicoree|endive)\w*\b',x):
        return .37,81,'UNKNOWN'  # no verified crop-specific coefficient
    for keys,c,cn,cl in [(['prairie','herbe','grass'],.01,61,'grass'),(['mais','maize'],.61,81,'maize'),(['pomme de terre','potato'],.64,81,'potato'),(['betterave','beet'],.41,81,'beet'),(['colza','rapeseed'],.38,81,'rape'),(['froment','orge','epeautre','avoine','seigle','triticale','cereale','cereal'],.39,76,'cereal'),(['lin','chanvre','wintervlas','zomervlas','vlas'],.68,81,'flax_hemp')]:
        if any(re.search(r'\b'+re.escape(k)+r'\w*\b',x) for k in keys): return c,cn,cl
    return .37,81,'UNKNOWN'
p[['C_crop','CN_base','crop_group']]=pd.DataFrame([crop_params(x) for x in p.crop_label],index=p.index)
p['LANDUSE']=np.where(p.crop_group=='grass',-4,-9999).astype(int)
p['C_reduct']=0.0
p['parcel_id']=np.arange(1,len(p)+1)
p['area_ha']=p.geometry.area/10000
display(p.groupby(['crop_label','crop_group','C_crop','CN_base'],dropna=False).area_ha.agg(['count','sum']).reset_index())
p['C_source']=np.where(p.crop_group=='UNKNOWN','NVE other-land fallback (analogy)',np.where(p.crop_group=='grass','NVE grass value; perennial/vegetated-cover analogy where applicable','Walloon NVE GISER / other-cover table'))
p['CN_source']=np.where(p.crop_group=='grass','TR-55 2-2c good pasture, B',np.where(p.crop_group=='cereal','TR-55 2-2b poor straight-row small grain, B','TR-55 2-2b poor straight-row crops, B (analogy)'))
p.drop(columns='geometry').to_csv(PACKAGE/'parcel_parameters.csv',index=False)
(PACKAGE/'parameter_references.txt').write_text("""C: Walloon NVE GISER crop and other-cover tables:
https://natuurwaardeverkenner.marvintest.vito.be/docs/manual_FR/regulerende_diensten/Reguleren_stromen/erosie/
CN: USDA SCS (1986), TR-55, tables 2-2a-c, reproduced by USACE:
https://www.hec.usace.army.mil/confluence/hmsdocs/hmstrm/cn-tables
Alternative European C methodology: Panagos et al. (2015), Land Use Policy 48, 38-50:
https://esdac.jrc.ec.europa.eu/content/cover-management-factor-c-factor-eu
Model documentation: https://watem-sedem.github.io/watem-sedem/input.html
C values are annual; storm reuse is provisional. HSG B uniform; crop CN analogies and unknown crop fallbacks not calibrated. K and kTC provisional.
""")
p.to_file(PACKAGE/'parcels.gpkg',driver='GPKG')
p[['LANDUSE','C_crop','C_reduct','parcel_id','geometry']].to_file(PACKAGE/'parcels.shp')
if (p.crop_group=='UNKNOWN').any(): print('WARNING: unknown crop labels use provisional C/CN; inspect CSV before scientific use.')
# Crop and management tables for future student interface
pd.DataFrame([('baseline',1,0),('CN_only',1,-10),('C_only',.6,0),('combined',.6,-10)],columns=['scenario','C_multiplier','CN_delta']).to_csv(PACKAGE/'management_scenarios.csv',index=False)

# crop domain subset with 2-cell margin, retaining grid alignment
r0,r1=max(0,np.where(mask)[0].min()-10),min(mask.shape[0],np.where(mask)[0].max()+11)
c0,c1=max(0,np.where(mask)[1].min()-10),min(mask.shape[1],np.where(mask)[1].max()+11)
sl=np.s_[r0:r1,c0:c1]; subtr=tr*rasterio.Affine.translation(c0,r0); m=mask[sl]; cover=lc[sl]
# WaTEM landuse codes: 10 generic crop, -3 forest, -4 pasture, -5 pond, -2 infrastructure
bg=np.full(m.shape,10,dtype='int16')
for k,v in {10:-3,20:-4,30:-4,40:10,50:-2,60:10,70:10,80:-5,90:-4,95:-4,100:-4}.items(): bg[cover==k]=v
bg[~m]=-9999
write_raster(PACKAGE/'dtm_original.tif',dem_original[sl],transform=subtr)
write_raster(PACKAGE/'dtm.tif',dem_hydro[sl],transform=subtr)
write_raster(PACKAGE/'dtm_difference.tif',(dem_hydro-dem_original)[sl],transform=subtr)
write_raster(PACKAGE/'basemap_landuse.tif',bg,'int16',subtr)
write_raster(PACKAGE/'K3.tif',np.full(m.shape,30,dtype='int16'),'int16',subtr)
write_raster(PACKAGE/'HSG_provisional.tif',np.where(m,2,-9999),'int16',subtr)
for name in ['roads','buildings','water']:
    g=vectors[name]
    if len(g):
        g=g[['geometry']].copy()
        if name=='roads': g['paved']=-2
        g.to_file(PACKAGE/(('infrastructure' if name=='buildings' else name)+'.shp'))
# Model river initiation threshold (physical area, independent of display threshold).
MODEL_RIVER_HA=25 #@param {type:'number'}
assert 0<MODEL_RIVER_HA<=float(upstream_ha[row,col]),'Channel threshold must not exceed outlet contributing area'
model_river=mask & (upstream_ha>=MODEL_RIVER_HA)
assert model_river[row,col]
# Every model channel must follow the conditioned D8 network to the chosen outlet.
segments=[]
for ri,ci in zip(*np.where(model_river)):
    if (ri,ci)==(row,col): continue
    dy,dx=D8_OFFSETS[int(fd[ri,ci])]; rt,ct=ri+dy,ci+dx
    assert model_river[rt,ct],'Thresholded D8 channel is discontinuous'
    segments.append(LineString([(xx[ri,ci],yy[ri,ci]),(xx[rt,ct],yy[rt,ct])]))
# Export exact channel raster and GIS line geometry; native code uses raster, not vector re-rasterization.
write_raster(PACKAGE/'model_river.tif',model_river[sl].astype('int16'),'int16',subtr)
write_raster(PACKAGE/'upstream_area_ha.tif',np.where(m,upstream_ha[sl],-9999).astype('float32'),transform=subtr)
if not segments:
    # Isolated outlet is not a useful channel network.
    raise ValueError('Threshold produces only the outlet cell; reduce MODEL_RIVER_HA.')
gpd.GeoDataFrame(geometry=segments,crs=CRS).to_file(PACKAGE/'river.shp')
if len(vectors['river']): vectors['river'][['geometry']].to_file(PACKAGE/'rhw_river.shp')
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
# Categorical palette for the model, not a continuous numerical colour scale.
LC_COLORS=['#fff0a6','#1c6338','#e99b45','#cf3b39','#69bce5','#1769b3','#91ba60']
LC_NAMES=['Cropland','Forest','Grass / pasture','Built-up / roads','Standing water','Model river','Grass strip']
def plot_classes(ax,arr,active,channel=None,title='Land cover'):
    a=np.zeros(arr.shape,dtype='uint8')
    for code,idx in [(-3,1),(-4,2),(-2,3),(-5,4),(-1,5),(-6,6)]: a[arr==code]=idx
    if channel is not None: a[channel]=5
    cmap=ListedColormap(LC_COLORS); cmap.set_bad((0,0,0,0))
    ax.imshow(np.ma.masked_where(~active,a),cmap=cmap,norm=BoundaryNorm(np.arange(-.5,7.5),7),interpolation='nearest')
    ax.set_title(title); ax.set_axis_off()
    ax.legend(handles=[Patch(facecolor=LC_COLORS[i],label=LC_NAMES[i]) for i in range(7) if np.any(active&(a==i))],loc='upper left',bbox_to_anchor=(1.01,1),fontsize=8,frameon=False)
# C-factor visualization with explicit zero and logarithmically spaced positive classes.
from matplotlib.colors import ListedColormap, BoundaryNorm
C_BOUNDS=[-0.000001,0.0000001,0.005,0.02,0.1,0.3,0.5,0.7,1.000001]
C_COLORS=['#7295ad','#164d31','#76a84f','#c5d96f','#f5e49a','#e9a34e','#c85b32','#8c1c27']
C_LABELS=['0 (non-eroding)','0–0.005 (forest range)','0.005–0.02 (grass range)','0.02–0.1','0.1–0.3','0.3–0.5','0.5–0.7','0.7–1']
def plot_cfactor(ax,values,active,title):
    values=np.asarray(values)
    valid=np.isfinite(values)&(values>=0)&(values<=1)
    assert np.all(valid[active]),'Active cells have missing or invalid C factor'
    cmap=ListedColormap(C_COLORS); cmap.set_bad((0,0,0,0))
    ax.imshow(np.ma.masked_where(~active,values),cmap=cmap,norm=BoundaryNorm(C_BOUNDS,len(C_COLORS)),interpolation='nearest')
    ax.set_title(title); ax.set_axis_off()
    ax.legend(handles=[Patch(facecolor=color,label=label) for color,label in zip(C_COLORS,C_LABELS)],loc='upper left',bbox_to_anchor=(1.01,1),fontsize=8,frameon=False)
# Preliminary complete C map, before exact native infrastructure/parcel mosaic.
# All final values are independently generated and checked in the next cell.
from rasterio.features import rasterize
pre_crop=rasterize([(g,float(v)) for g,v in zip(p.geometry,p.C_crop) if g is not None and not g.is_empty],out_shape=m.shape,transform=subtr,fill=-9999,dtype='float32')
pre_grass=rasterize([(g,1) for g in p.loc[p.crop_group=='grass','geometry'] if g is not None and not g.is_empty],out_shape=m.shape,transform=subtr,fill=0,dtype='uint8') if (p.crop_group=='grass').any() else np.zeros(m.shape,dtype='uint8')
pre_c=np.full(m.shape,-9999,dtype='float32')
pre_c[m]=.37
for code,val in [(-3,.001),(-4,.01),(-2,0),(-5,0)]: pre_c[m&(bg==code)]=val
has_parcel=m&(pre_crop>0)&(pre_crop<=1)
pre_c[has_parcel]=pre_crop[has_parcel]
pre_c[m&(pre_grass==1)]=.01
pre_c[model_river[sl]]=0
write_raster(PACKAGE/'C_preliminary.tif',pre_c,transform=subtr)
fig,ax=plt.subplots(2,3,figsize=(17,10),constrained_layout=True)
for a,data,title in [(ax[0,0],dem_hydro[sl],'Conditioned elevation (m)'),(ax[0,2],np.where(m,30,np.nan),'Provisional K')]:
    im=a.imshow(np.where(m,data,np.nan)); a.set_title(title); fig.colorbar(im,ax=a,shrink=.7)
plot_classes(ax[0,1],bg,m,model_river[sl],title='Background land cover + model channels')
plot_cfactor(ax[1,0],pre_c,m,'Preliminary complete C factor (before native infrastructure)')
vectors['soils'].plot(ax=ax[1,1],facecolor='none',edgecolor='black'); ax[1,1].set_title('Actual CNSW soil polygons')
boundary.boundary.plot(ax=ax[1,2],color='black')
for name,color in [('roads','grey'),('river','blue'),('buildings','red')]:
    if len(vectors[name]): vectors[name].plot(ax=ax[1,2],color=color)
gpd.GeoDataFrame(geometry=segments,crs=CRS).plot(ax=ax[1,2],color='#00a5ff',linewidth=1.4)
ax[1,2].set_title('Infrastructure, RHW and D8 model channels')
fig.savefig(PACKAGE/'input_atlas.png',dpi=180); plt.show()
manifest=dict(seed=SEED,year=YEAR,area_ha=float(m.sum()*RES**2/10000),outlet_lonlat=list(outlet.to_crs(4326).geometry.iloc[0].coords)[0],resolution=RES,dem=dem_sources,landcover=wc_url,vector_services=sources,K_provisional=30,HSG_provisional='B',unknown_crop_count=int((p.crop_group=='UNKNOWN').sum()),hydro_conditioning={'burn_depth_m':BURN_DEPTH,'min_slope_deg':MIN_SLOPE,'river_source':sources['river'],'method':'SAGA burn method 0 then Wang-Liu fill; D8 closure check'},model_river_threshold_ha=MODEL_RIVER_HA,model_river_cells=int(model_river.sum()),scientific_status='pilot assumptions; not calibrated')
(PACKAGE/'metadata.json').write_text(json.dumps(manifest,indent=2))
if VERBOSE: print('Saved prepared inputs:',PACKAGE)

# Save a human-readable data lineage table with the GIS reference layers.
lineage=[
 ('Elevation','Copernicus GLO-30 or supplied SPW DEM','; '.join(dem_sources),'Bilinear resampling to 20 m; RHW stream burning and sink filling','dtm_original.tif; dtm.tif'),
 ('Background cover','ESA WorldCover 2021 v200','; '.join(wc_url),'Nearest-neighbour resampling and model class crosswalk','basemap_landuse.tif; native PRC'),
 ('Crop parcels','SPW SIGEC '+str(YEAR),sources['parcels'],'Clip, decode crop labels, assign provisional C and CN; rasterize','parcels.gpkg; parcel_parameters.csv; native PRC/C/CN'),
 ('Soils','SPW CNSW',sources['soils'],'Clip for interpretation; no soil-specific K/HSG derivation yet','soils_source.gpkg'),
 ('Rivers','SPW RHW',sources['river'],'Burn mapped streams into DEM; D8 accumulation threshold defines model channels','rhw_river.shp; model_river.tif; native PRC'),
 ('Roads','SPW PICC',sources['roads'],'Clip and rasterize infrastructure','roads.shp; native PRC'),
 ('Buildings','SPW PICC',sources['buildings'],'Clip and rasterize infrastructure','infrastructure.shp; native PRC'),
 ('Standing water','SPW PICC',sources['water'],'Clip; exclude pools where identified; rasterize','water.shp; native PRC'),
 ('K / HSG','Teaching assumptions','Not downloaded','Constant K=30; HSG B provisional','K3.tif; HSG_provisional.tif'),
 ('Rainfall','Synthetic 40 mm event','Not downloaded','Fixed temporal shape; event erosivity; CN model','rainfall text; INI'),
 ('Model settings',SETTINGS_SOURCE,str(REV),'Local INI; override relevant options; no tutorial geography used','native INI')]
lineage_df=pd.DataFrame(lineage,columns=['input','source','url_or_reference','conversion','model_products'])
display(lineage_df); lineage_df.to_csv(PACKAGE/'data_lineage.csv',index=False)
(PACKAGE/'cn_parameters.json').write_text(json.dumps(dict(alpha=CN_ALPHA,beta=CN_BETA,AR5_mm=CN_AR5_MM,status='provisional; corrections off by default; source uses AR5*beta without /10'),indent=2))


from pywatemsedem.catchment import Catchment
from pywatemsedem.choices import Choices, Options, Parameters, Extensions, ExtensionsParameters, Output
from pywatemsedem.scenario import Scenario
import configparser
assert (PACKAGE/'model_river.tif').exists(),'Prepare D8 model channel raster first.'
catchment=Catchment(SITE_ID,PACKAGE/'catchment.shp',PACKAGE/'dtm.tif',RES,31370,-9999,results_folder=WORK)
# Replace the polygon-derived wrapper mask by the original D8 raster on its exact grid.
from rasterio.warp import reproject, Resampling
from pywatemsedem.geo.rasters import RasterMemory
rp=catchment.rp.rasterio_profile
shape0=catchment.mask.arr.shape
# Both grids share the same CRS, resolution and pixel origin; slice directly.
offset_col,offset_row=(~tr)*(rp['transform'].c,rp['transform'].f)
assert np.allclose([offset_col,offset_row],np.round([offset_col,offset_row]),atol=1e-5)
assert np.allclose([tr.a,tr.b,tr.d,tr.e],[rp['transform'].a,rp['transform'].b,rp['transform'].d,rp['transform'].e])
offset_col,offset_row=int(round(offset_col)),int(round(offset_row))
assert 0<=offset_row and 0<=offset_col and offset_row+shape0[0]<=mask.shape[0] and offset_col+shape0[1]<=mask.shape[1]
def align_binary(a):
    return a[offset_row:offset_row+shape0[0],offset_col:offset_col+shape0[1]].astype('uint8').copy()
d8_grid=align_binary(mask)
model_grid=align_binary(model_river)
old_inside=np.asarray(catchment.mask.arr)==1
inside0=d8_grid==1
assert int(inside0.sum())==int(mask.sum()),'Wrapper extent loses D8 cells'
assert int(model_grid.sum())==int(model_river.sum()),'Wrapper extent loses D8 river cells'
assert not np.any((model_grid==1)&~inside0)
print('Polygon rasterization difference: omitted D8 cells',int((inside0&~old_inside).sum()),'extra cells',int((old_inside&~inside0).sum()))
catchment._mask=RasterMemory(np.where(inside0,1,catchment.rp.nodata).astype('int16'),catchment.rp)
catchment.mask.arr_bin=d8_grid.copy()
catchment.mask.write(catchment.rasterfile_mask,format='tiff',dtype=np.int16,nodata=catchment.rp.nodata)
assert np.array_equal(catchment.mask.arr==1,inside0)
with rasterio.open(catchment.rasterfile_mask) as ds:
    assert np.array_equal(ds.read(1)==1,inside0),'Disk mask differs from D8 watershed'
catchment.kfactor=PACKAGE/'K3.tif'
catchment.landuse=PACKAGE/'basemap_landuse.tif'
catchment.river=model_grid.astype('int32')
assert np.array_equal(catchment.river.arr==-1,model_grid==1),'Wrapper changed the supplied river raster'
catchment.kfactor.arr[:]=30
print('D8 model channel cells:',int(model_grid.sum()),'| threshold:',MODEL_RIVER_HA,'ha')
for prop,name in [('vct_infrastructure_buildings','infrastructure'),('vct_infrastructure_roads','roads'),('vct_water','water')]:
    path=PACKAGE/f'{name}.shp'
    if path.exists(): setattr(catchment,prop,path)
default_choices=inputdata_folder/'userchoices.ini'
def settings():
    o=Options(); o.read_values_from_ini(default_choices)
    par=Parameters(); par.read_values_from_ini(default_choices)
    ext=Extensions(); ext.read_values_from_ini(default_choices)
    out=Output(); out.read_values_from_ini(default_choices)
    ep=ExtensionsParameters(ext); ep.apply_defaults()
    for v in (ep.ktc_low,ep.ktc_high,ep.ktc_limit): v.read_value_from_ini(default_choices)
    return Choices(o,par,ext,ep,out)
rain=np.array([0,2,4,7,9,8,5,3,2,0],dtype=float)
rainfile=WORK/'storm.txt'; np.savetxt(rainfile,np.column_stack([np.arange(len(rain))*10,rain]),fmt=['%d','%.3f'],delimiter='\t')
intensity=rain[1:]*6; R=float(np.sum(.29*(1-.72*np.exp(-.05*intensity))*rain[1:])*np.max(np.convolve(rain[1:],np.ones(3),'valid'))*2)
scenarios={}
for nr,mode in enumerate(['annual','event'],1):
    s=Scenario(catchment,YEAR,nr,settings())
    # Raster-only river network; no segment topology or routing tables are supplied.
    s.choices.extensions.river_routing=False
    s.choices.extensions.output_per_river_segment=False
    for flag in ['write_aspect','write_ls_factor','write_upstream_area','write_slope','write_routing_table','write_rusle','write_sediment_export','write_water_erosion']:
        setattr(s.choices.output,flag,True)
    if mode=='event':
        s.choices.extensions.curve_number=True; s.choices.extensions.convert_output=False
        s.choices.parameters.r_factor=R
        ep=s.choices.extensionparameters; ep.alpha=CN_ALPHA; ep.beta=CN_BETA; ep.antecedent_rainfall=CN_AR5_MM; ep.stream_velocity=.2; ep.desired_timestep=60; ep.endtime_model=720
        s.choices.output.write_total_runoff=True; s.choices.output.write_rainfall_excess=True
    s.vct_parcels=PACKAGE/'parcels.shp'
    composite=s.create_composite_landuse()
    # Wrapper uses 32767 for unclassified active cells and background cropland.
    # Replace only this reserved/gap code with actual aligned WorldCover classes.
    # Use a distinct positive ID for non-SIGEC generic cropland.
    gaps=inside0 & (composite==32767)
    background=np.full(lc.shape,10,dtype='int16')
    for code,value in {10:-3,20:-4,30:-4,40:10,50:-2,60:10,70:10,80:-5,90:-4,95:-4,100:-4}.items():
        background[lc==code]=value
    bg_native=background[offset_row:offset_row+shape0[0],offset_col:offset_col+shape0[1]]
    generic_id=int(len(p)+1)
    assert generic_id<=32757
    fallback=np.where(bg_native==10,generic_id,bg_native)
    composite[gaps]=fallback[gaps]
    assert not np.any(inside0 & ((composite<=-9999)|(composite==0)|(composite>32757)))
    s.composite_landuse=composite
    if mode=='annual':
        print('Background land-cover gap repair:',int(gaps.sum()),'cells; generic crop ID:',generic_id)
        pd.DataFrame([{'replacement_code':int(v),'cells':int((gaps & (fallback==v)).sum())} for v in np.unique(fallback[gaps])]).to_csv(PACKAGE/'background_gap_repair.csv',index=False)
    # Build C from the FINAL PRC classes and parcel crop factors on the same grid.
    lu=np.asarray(s.composite_landuse.arr)
    crop_c=rasterize([(g,float(v)) for g,v in zip(p.geometry,p.C_crop) if g is not None and not g.is_empty],out_shape=lu.shape,transform=rp['transform'],fill=-9999,dtype='float32')
    c=np.full(lu.shape,-9999,dtype='float32')
    crop=inside0 & (lu>0)
    covered=crop & np.isfinite(crop_c) & (crop_c>0)&(crop_c<=1)
    c[crop]=.37; c[covered]=crop_c[covered]
    for code,value in {-1:0,-2:0,-3:.001,-4:.01,-5:0,-6:.01}.items(): c[inside0 & (lu==code)]=value
    assert np.all(np.isfinite(c[inside0])&(c[inside0]>=0)&(c[inside0]<=1)),'Missing/invalid C values'
    assert np.all(c[inside0 & ~np.isin(lu,[-1,-2,-5])]>0),'Unexpected zero C on vegetated or crop cells'
    s.cfactor=c
    print('C-factor coverage:',int(inside0.sum()),'valid active cells; crop-specific:',int(covered.sum()),'default crop C=0.37:',int((crop&~covered).sum()))
    if mode=='annual':
        fig,ax=plt.subplots(figsize=(10,7),constrained_layout=True)
        plot_cfactor(ax,c,inside0,'Complete model C factor (before native export)'); plt.show()
    assert np.array_equal(s.composite_landuse.arr==-1,model_grid==1),'Composite land use changed model channel cells'
    assert np.all(s.cfactor.arr[model_grid==1]==0),'Channel C factor must be zero'
    ep=s.choices.extensionparameters
    s.ktc=s.create_ktc(ep.ktc_low.value,ep.ktc_high.value,ep.ktc_limit.value,not s.choices.extensions.create_ktc_map.value)
    if mode=='event':
        lu=np.asarray(s.composite_landuse.arr); inside=np.asarray(catchment.mask.arr)==1
        cn=np.zeros(lu.shape,dtype='float32')
        cn[inside & (lu>0)]=81
        # Use model grid profile, not the source raster shape.
        pr=s.rp.rasterio_profile
        crop_cn=rasterize([(g,float(v)) for g,v in zip(p.geometry,p.CN_base)],out_shape=lu.shape,transform=pr['transform'],fill=0,dtype='float32')
        use=inside & (lu>0)&(crop_cn>0); cn[use]=crop_cn[use]
        for k,v in {-1:100,-2:98,-3:55,-4:61,-5:100,-6:61}.items(): cn[inside & (lu==k)]=v
        assert np.all((cn[inside]>0)&(cn[inside]<=100)),np.unique(lu[inside & (cn==0)])
        s.cn=cn; s.rainfall=rainfile
    # Explicitly refill the entire wrapper rectangular DEM from the full conditioned search grid.
    from rasterio.warp import reproject, Resampling
    model_profile=s.rp.rasterio_profile
    full=np.full(s.composite_landuse.arr.shape,np.nan,dtype='float32')
    reproject(dem_hydro,full,src_transform=tr,src_crs=CRS,dst_transform=model_profile['transform'],dst_crs=CRS,dst_nodata=np.nan,resampling=Resampling.nearest)
    assert np.isfinite(full).all() and (full>-1000).all(), 'Model rectangular window lacks valid DEM coverage'
    catchment.dtm.arr=full.copy()
    s.prepare_input_files(); s.create_ini_file()
    # Native Pascal reads these two independent switches. Raster-only channels
    # do not need segment IDs, adjacency or upstream segment tables.
    cfg_river=configparser.ConfigParser(interpolation=None); cfg_river.optionxform=str; cfg_river.read(s.ini)
    ext_section=next(x for x in cfg_river.sections() if x.lower()=='extensions')
    for flag in ['River Routing','Output per river segment']:
        key=next((k for k in cfg_river[ext_section] if k.lower()==flag.lower()),flag)
        cfg_river[ext_section][key]='0'
    with s.ini.open('w') as f: cfg_river.write(f)
    if VERBOSE: print('Raster-only rivers: segment routing and per-segment output disabled.')
    # Verify actual exported DTM, PRC and river pixels before running.
    native_dir=s.sfolder.wsinput_folder
    ini_check=configparser.ConfigParser(interpolation=None); ini_check.read(s.ini)
    dtm_candidates=[v for sec in ini_check.sections() for k,v in ini_check[sec].items() if 'dtm' in k.lower() and v.lower().endswith(('.rst','.tif'))]
    assert len(dtm_candidates)==1,dtm_candidates
    dtm_path=native_dir/Path(dtm_candidates[0]).name
    with rasterio.open(dtm_path) as ds:
        before=ds.read(1,masked=True)
        native_transform=ds.transform
        native_crs=ds.crs
        if VERBOSE: print('Wrapper DEM before repair:', 'nodata',ds.nodata,'masked',int(np.ma.getmaskarray(before).sum()),'raw range',float(np.min(before.data)),float(np.max(before.data)))
        assert ds.shape==full.shape and native_transform.almost_equals(model_profile['transform']), 'Native grid alignment mismatch'
    # Write directly using GDAL RST CreateCopy via rasterio, including a new RDC.
    # Verify the round trip and grid alignment before replacing native files.
    from rasterio.shutil import copy as rio_copy
    tif=WORK/f'{mode}_full_dtm.tif'
    with rasterio.open(tif,'w',driver='GTiff',height=full.shape[0],width=full.shape[1],count=1,dtype='float32',crs=CRS,transform=model_profile['transform'],nodata=-99999.) as ds:
        ds.write(full,1)
    temp=WORK/f'{mode}_full_dtm.rst'
    rio_copy(tif,temp,driver='RST')
    with rasterio.open(temp) as ds:
        raw=ds.read(1,masked=True)
        assert ds.dtypes[0]=='float32' and not np.ma.getmaskarray(raw).any() and np.allclose(raw.data,full,atol=.001), 'Direct RST creation failed'
        assert ds.transform.almost_equals(native_transform)
    # Replace both raster and header; delete stale GDAL auxiliary metadata.
    for suffix in ['.rst','.rdc']:
        shutil.copy2(temp.with_suffix(suffix),dtm_path.with_suffix(suffix))
    for suffix in ['.aux.xml','.msk','.ovr']:
        Path(str(dtm_path)+suffix).unlink(missing_ok=True)
    with rasterio.open(dtm_path) as ds:
        native_dem=ds.read(1,masked=True)
        if VERBOSE: print('Repaired native DEM:',ds.shape,ds.dtypes[0],'nodata',ds.nodata,'masked',int(np.ma.getmaskarray(native_dem).sum()),'range',float(native_dem.data.min()),float(native_dem.data.max()))
        assert ds.transform.almost_equals(native_transform) and ds.shape==full.shape
        assert not np.ma.getmaskarray(native_dem).any() and np.isfinite(native_dem.data).all() and (native_dem.data>-1000).all(), 'Native DEM contains invalid elevations'
        assert np.allclose(native_dem.data,full,atol=.001), 'Native DEM differs from full window'
    lu=np.asarray(s.composite_landuse.arr); inside=np.asarray(catchment.mask.arr)==1
    print(mode,'model cells',inside.sum(),'river cells',np.sum(inside & (lu==-1)),'DEM range',float(full.min()),float(full.max()))
    assert np.any(inside & (lu==-1)), 'No native river cells in basin'
    if VERBOSE: print('Outside-domain parcel codes:',np.unique(lu[~inside]))
    # Native executable recognizes only PRC=0 as outside; -9999 is NOT an outside code.
    ini_files={k:v for sec in ini_check.sections() for k,v in ini_check[sec].items() if v.lower().endswith('.rst')}
    if VERBOSE: print('INI raster references:',ini_files)
    def native_read(key):
        matches=[v for k,v in ini_files.items() if key(k)]
        assert len(matches)==1,matches
        with rasterio.open(native_dir/Path(matches[0]).name) as ds: return ds.read(1),ds.transform
    prc_matches=[v for k,v in ini_files.items() if 'parcel' in k.lower() and 'filename' in k.lower()]
    if len(prc_matches)!=1:
        prc_matches=[f.name for f in native_dir.glob('*.rst') if 'parcelmosaic' in f.stem.lower() or f.stem.lower() in ('prc','perceelskaart')]
    assert len(prc_matches)==1,('Cannot locate executable parcel raster',ini_files)
    prc_path=native_dir/Path(prc_matches[0]).name
    with rasterio.open(prc_path) as ds: native_prc=ds.read(1); prc_transform=ds.transform
    if VERBOSE: print('Native PRC outside raw codes BEFORE:',np.unique(native_prc[~inside]),'inside codes:',np.unique(native_prc[inside]))
    assert prc_transform.almost_equals(native_transform)
    # Rebuild native PRC from actual model array and explicit outside=0, not NoData.
    prc=np.where(inside,lu,0).astype('int16')
    assert np.all((prc[inside]>0)|(np.isin(prc[inside],[-1,-2,-3,-4,-5,-6])))
    assert np.max(prc)<=32757,'Parcel IDs exceed model range'
    prctif=WORK/f'{mode}_prc.tif'; prcrst=WORK/f'{mode}_prc.rst'
    with rasterio.open(prctif,'w',driver='GTiff',height=prc.shape[0],width=prc.shape[1],count=1,dtype='int16',crs=CRS,transform=native_transform,nodata=-9999) as ds: ds.write(prc,1)
    rio_copy(prctif,prcrst,driver='RST')
    for suffix in ['.rst','.rdc']: shutil.copy2(prcrst.with_suffix(suffix),prc_path.with_suffix(suffix))
    for suffix in ['.aux.xml','.msk','.ovr']: Path(str(prc_path)+suffix).unlink(missing_ok=True)
    with rasterio.open(prc_path) as ds:
        native_prc=ds.read(1); assert np.array_equal(native_prc,prc)
    if VERBOSE: print('Native PRC outside raw codes AFTER:',np.unique(native_prc[~inside]),'native river cells:',int((native_prc==-1).sum()))
    assert np.all(native_prc[~inside]==0)
    assert np.array_equal(native_prc==-1,model_grid==1),'Native PRC changed the D8 model river raster'
    rr_out,cc_out=rasterio.transform.rowcol(native_transform,*outxy)
    assert native_prc[rr_out,cc_out]==-1,'Confirmed outlet is not a native river cell'
    # Constant provisional K is defined on the entire model grid, not masked by
    # a separately rasterized watershed polygon. PRC alone defines active cells.
    k_matches=[v for k,v in ini_files.items() if k.lower()=='k factor filename']
    assert len(k_matches)==1,k_matches
    kpath=native_dir/Path(k_matches[0]).name
    with rasterio.open(kpath) as ds:
        k_before=ds.read(1)
        assert ds.shape==full.shape and ds.transform.almost_equals(native_transform)
    invalid_k=inside & (~np.isfinite(k_before) | (k_before<0))
    if VERBOSE: print(mode,'K QA: invalid active cells before native normalization:',int(invalid_k.sum()))
    # Compare against the original D8 mask using exact integer grid indexing.
    # No reprojection: zero is a valid binary class, not NoData.
    inv=~tr
    oc,orr=inv*(native_transform.c,native_transform.f)
    assert np.allclose([oc,orr],np.round([oc,orr]),atol=1e-5)
    assert np.allclose([native_transform.a,native_transform.b,native_transform.d,native_transform.e],[tr.a,tr.b,tr.d,tr.e])
    oc,orr=int(round(oc)),int(round(orr))
    nr0,nc0=inside.shape
    assert 0<=orr and 0<=oc and orr+nr0<=mask.shape[0] and oc+nc0<=mask.shape[1]
    exact_d8=mask[orr:orr+nr0,oc:oc+nc0].astype(bool)
    assert int(exact_d8.sum())==int(mask.sum()),'Native window truncates D8 basin'
    assert np.array_equal(inside,exact_d8),'Actual model mask differs from original D8 mask'
    assert np.array_equal(native_prc!=0,exact_d8),'Native PRC domain differs from D8'
    # The wrapper's Idrisi export can turn valid zeros into ones. Write the native
    # mask explicitly with -9999 outside, and verify the raw cells after export.
    mask_path=native_dir/Path(ini_files['shapefile catchment']).name
    mask_tif=WORK/f'{mode}_exact_mask.tif'; mask_rst=WORK/f'{mode}_exact_mask.rst'
    exact_values=np.where(exact_d8,1,-9999).astype('int16')
    with rasterio.open(mask_tif,'w',driver='GTiff',height=nr0,width=nc0,count=1,dtype='int16',crs=CRS,transform=native_transform,nodata=-9999) as ds:
        ds.write(exact_values,1)
    rio_copy(mask_tif,mask_rst,driver='RST')
    for suffix in ['.rst','.rdc']: shutil.copy2(mask_rst.with_suffix(suffix),mask_path.with_suffix(suffix))
    for suffix in ['.aux.xml','.msk','.ovr']: Path(str(mask_path)+suffix).unlink(missing_ok=True)
    with rasterio.open(mask_path) as ds:
        assert ds.transform.almost_equals(native_transform)
        assert np.array_equal(ds.read(1),exact_values),'Native mask export changed values'
    print('D8 / wrapper / native PRC / native mask:',int(exact_d8.sum()),'active cells; exact agreement')
    if invalid_k.any():
        print('WARNING: wrapper export introduced invalid K; full-grid constant normalization follows.')
    k_fixed=np.full(full.shape,30,dtype='int16')
    ktif=WORK/f'{mode}_k_full.tif'; krst=WORK/f'{mode}_k_full.rst'
    with rasterio.open(ktif,'w',driver='GTiff',height=full.shape[0],width=full.shape[1],count=1,dtype='int16',crs=CRS,transform=native_transform,nodata=-9999) as ds: ds.write(k_fixed,1)
    rio_copy(ktif,krst,driver='RST')
    for suffix in ['.rst','.rdc']: shutil.copy2(krst.with_suffix(suffix),kpath.with_suffix(suffix))
    for suffix in ['.aux.xml','.msk','.ovr']: Path(str(kpath)+suffix).unlink(missing_ok=True)
    with rasterio.open(kpath) as ds:
        k_after=ds.read(1,masked=True)
        assert ds.transform.almost_equals(native_transform) and ds.dtypes[0]=='int16'
        assert not np.ma.getmaskarray(k_after).any() and np.array_equal(k_after.data,k_fixed)
    print(mode,'K repair: all grid cells = 30 kg h/(MJ mm); invalid active cells = 0')
    # Verify raw numerical parameter values in the active model domain.
    for k,v in ini_files.items():
        if not any(x in k.lower() for x in ['c factor','c-factor','k factor','k-factor','p factor','p-factor','ktc']): continue
        with rasterio.open(native_dir/Path(v).name) as ds:
            a=ds.read(1); vv=a[inside & (native_prc!=-1)]
            if VERBOSE: print(k,'active min/max',float(vv.min()),float(vv.max()),'negative cells',int((vv<0).sum()))
            assert np.isfinite(vv).all() and (vv>=0).all(),f'Invalid active {k} values'
            if 'c factor' in k.lower() or 'c-factor' in k.lower():
                assert np.allclose(a[inside],c[inside],atol=1e-6),'Native C export changed values'
            if 'p factor' in k.lower() or 'p-factor' in k.lower(): assert (vv>0).all(),'P=0 triggers invalid RUSLE sentinel in Pascal 5.0.3'
    # Read back the actual C raster passed to the executable, not the preview.
    c_refs=[v for k,v in ini_files.items() if 'c factor' in k.lower() or 'c-factor' in k.lower()]
    assert len(c_refs)==1,('Cannot identify native C raster',ini_files)
    with rasterio.open(native_dir/Path(c_refs[0]).name) as ds:
        native_c=ds.read(1,masked=True)
        assert ds.transform.almost_equals(native_transform) and ds.shape==inside.shape
        assert not np.ma.getmaskarray(native_c)[inside].any(),'Native C has missing active cells'
        c_actual=native_c.data
        assert np.allclose(c_actual[inside],c[inside],atol=1e-6),'Native C export changed expected values'
        assert np.all(c_actual[inside & ~np.isin(native_prc,[-1,-2,-5])]>0)
    if mode=='annual':
        rows=[]
        for label,sel in [('Cropland',native_prc>0),('Forest',native_prc==-3),('Grassland',native_prc==-4),('Infrastructure',native_prc==-2),('River',native_prc==-1),('Standing water',native_prc==-5),('Grass strip',native_prc==-6)]:
            a=inside&sel; v=c_actual[a]
            rows.append(dict(land_cover=label,cells=int(a.sum()),area_ha=float(a.sum()*RES**2/10000),C_min=float(v.min()) if v.size else None,C_max=float(v.max()) if v.size else None,missing=int((a&np.ma.getmaskarray(native_c)).sum()),zero_C=int((v==0).sum())))
        qa_c=pd.DataFrame(rows); display(qa_c); qa_c.to_csv(PACKAGE/'native_C_coverage.csv',index=False)
        fig,ax=plt.subplots(figsize=(10,7),constrained_layout=True)
        plot_cfactor(ax,c_actual,inside,'Verified native C factor used by WaTEM/SEDEM')
        fig.savefig(PACKAGE/'native_C_factor.png',dpi=180,bbox_inches='tight'); plt.show()
    if mode=='annual':
        fig,ax=plt.subplots(figsize=(9,7)); plot_classes(ax,native_prc,inside,title='Final model land cover (PRC)'); fig.savefig(PACKAGE/'model_landcover.png',dpi=180,bbox_inches='tight'); plt.show()
    np.savez_compressed(WORK/f'{mode}_native_domain.npz',prc=native_prc,mask=inside,dem=full)
    assert not np.any(lu[~inside]>0), 'Agricultural parcel IDs outside model domain'
    if mode=='event':
        cfg=configparser.ConfigParser(interpolation=None); cfg.optionxform=str; cfg.read(s.ini)
        sec=next(x for x in cfg.sections() if x.lower()=='parameters extensions')
        for key,value in [('Endtime model',720),('Bulk density',int(s.choices.parameters.bulk_density.value))]:
            actual=next((x for x in cfg[sec] if x.lower()==key.lower()),key); cfg[sec][actual]=str(value)
        with s.ini.open('w') as f: cfg.write(f)
    # Route event runoff along the exact D8 channel links, using the native
    # forced-routing interface (not the separate river-segment extension).
    if mode=='event' and ROUTE_CHANNEL_RUNOFF:
        links=[]
        for ar,ac in zip(*np.where(model_river)):
            if (ar,ac)==(row,col): continue
            dr,dc=D8_OFFSETS[int(fd[ar,ac])]
            br,bc=ar+dr,ac+dc
            assert model_river[br,bc]
            links.append((ar-orr,ac-oc,br-orr,bc-oc))
        assert len(links)==int(model_grid.sum())-1
        assert all(native_prc[a,b]==-1 and native_prc[c_,d]==-1 for a,b,c_,d in links)
        outlet_arr=np.zeros(inside.shape,dtype='int16'); outlet_arr[rr_out,cc_out]=1
        otif=WORK/'explicit_outlet.tif'; orst=WORK/'explicit_outlet.rst'
        with rasterio.open(otif,'w',driver='GTiff',height=nr0,width=nc0,count=1,dtype='int16',crs=CRS,transform=native_transform,nodata=-9999) as ds: ds.write(outlet_arr,1)
        rio_copy(otif,orst,driver='RST')
        for extn in ['.rst','.rdc']: shutil.copy2(orst.with_suffix(extn),native_dir/('explicit_outlet'+extn))
        with rasterio.open(native_dir/'explicit_outlet.rst') as ds: assert np.array_equal(ds.read(1),outlet_arr)
        cfg_route=configparser.ConfigParser(interpolation=None); cfg_route.read(s.ini)
        cfg_route['Extensions']['Force Routing']='1'
        cfg_route['Extensions']['Manual outlet selection']='1'
        cfg_route['Files']['Outlet map filename']='explicit_outlet.rst'
        route_params=next(sec for sec in cfg_route.sections() if sec.lower()=='parameters extensions')
        cfg_route[route_params]['Number of Forced Routing']=str(len(links))
        for i,(a,b,c_,d) in enumerate(links,1):
            cfg_route[f'Forced Routing {i}']={'from row':str(a+1),'from col':str(b+1),'target row':str(c_+1),'target col':str(d+1)}
        with s.ini.open('w') as f: cfg_route.write(f)
        pd.DataFrame(links,columns=['from_row0','from_col0','target_row0','target_col0']).to_csv(PACKAGE/'channel_routing_links.csv',index=False)
        print('Explicit event channel routing:',len(links),'D8 links; confirmed outlet:',rr_out,cc_out)
    scenarios[mode]=s
    if VERBOSE: print(mode,s.ini)
assert set(scenarios)=={'annual','event'},'Both native scenarios must be prepared'
# Show final native settings, not the initial template.
settings_rows=[]
keys=[('Options','L model'),('Options','S model'),('Options','TC model'),('Parameters','R factor'),('Parameters','Bulk density'),('Parameters','Parcel connectivity cropland'),('Parameters','Parcel connectivity forest'),('Parameters','Parcel connectivity grasstrips'),('Parameters','Parcel trapping efficiency cropland'),('Parameters','Parcel trapping efficiency forest'),('Parameters','Parcel trapping efficiency pasture'),('Extensions','Curve Number'),('Extensions','Create ktc map'),('Extensions','River Routing'),('Extensions','Force Routing'),('Extensions','Manual outlet selection'),('Parameters extensions','Number of Forced Routing'),('Parameters extensions','Alpha'),('Parameters extensions','Beta'),('Parameters extensions','5-day antecedent rainfall'),('Parameters extensions','Stream velocity'),('Parameters extensions','Desired timestep for model'),('Parameters extensions','Endtime model'),('Output','Write rainfall excess'),('Output','Write total runoff')]
final_cfg={}
for mode in ('annual','event'):
    cfg_final=configparser.ConfigParser(interpolation=None); cfg_final.read(scenarios[mode].ini)
    # Normalize section names for display and validation; option names are already case-insensitive.
    normalized=configparser.ConfigParser(interpolation=None)
    for section in cfg_final.sections():
        normalized[section.lower()]=dict(cfg_final[section])
    final_cfg[mode]=normalized
for sec,key in keys:
    fallback='0' if sec=='Extensions' else '—'
    settings_rows.append({'Parameter':key,'Annual':final_cfg['annual'].get(sec.lower(),key,fallback=fallback),'40 mm event':final_cfg['event'].get(sec.lower(),key,fallback=fallback)})
effective=pd.DataFrame(settings_rows)
display(effective)
effective.to_csv(PACKAGE/'effective_model_settings.csv',index=False)
assert final_cfg['event'].getboolean('extensions','Curve Number')
assert final_cfg['event'].getboolean('output','Write total runoff')
assert final_cfg['event'].getboolean('output','Write rainfall excess')
assert not final_cfg['annual'].getboolean('extensions','Curve Number',fallback=False)
if ROUTE_CHANNEL_RUNOFF:
    assert final_cfg['event'].getboolean('extensions','Force Routing') and final_cfg['event'].getboolean('extensions','Manual outlet selection')
print('Both final configurations verified. kTC is supplied as a precomputed raster; segment routing is separate from forced channel links.')
DEST=OUTPUT/RUN_ID; DEST.mkdir(exist_ok=True)
for mode,s in scenarios.items(): shutil.copytree(s.sfolder.scenario_folder,DEST/mode,dirs_exist_ok=True)
print('Native inputs copied; INI paths refer to current local runtime. Regenerate on next runtime.')

## 6. Run baseline simulations and inspect the model atlas
Annual erosion plus synthetic 20, 40 and 60 mm events. The native input atlas shows the actual model rasters; separate output atlases show erosion, sediment transport and hydrology. Total runoff is cumulative water reaching each cell in m³, not a map of independent runoff production. Event channel routing uses explicit D8 links through the native forced-routing mechanism and a manual outlet raster; the separate river-segment extension remains off. Annual settings are unchanged. The new event routing requires numerical testing and water balance verification. The sediment event response remains erosivity-based.

In [ ]:
#@title 6 — Run and report all baseline simulations
# Run baseline annual and 40 mm event using the verified native inputs.
import urllib.request, sys
helper_dir=Path('/content/watem_student_helpers'); helper_dir.mkdir(exist_ok=True)
urllib.request.urlretrieve('https://raw.githubusercontent.com/kvanoost/Watem/50839661b3a72d2f7a78b7c3d0a4ba7019908cc7/watem_student_common.py',helper_dir/'watem_student_common.py')
sys.path.insert(0,str(helper_dir))
import watem_student_common as ws
from matplotlib.colors import LinearSegmentedColormap
# Negative = erosion (red/brown); positive = deposition (green).
redistribution_cmap=LinearSegmentedColormap.from_list('watem_redistribution',['#9c3022','#d97742','#fffdf5','#85bb79','#17613b'])
# The shared report uses RdBu only for its signed soil redistribution figure.
# Register a separate temporary style by wrapping the report's imshow call locally.
from unittest.mock import patch
from matplotlib.axes import Axes
_original_imshow=Axes.imshow
def _soil_imshow(self,*args,**kwargs):
    if kwargs.get('cmap')=='RdBu': kwargs['cmap']=redistribution_cmap
    return _original_imshow(self,*args,**kwargs)
def report_soil(folder,mode,rain_mm):
    with patch.object(Axes,'imshow',_soil_imshow):
        return ws.report(folder,mode,rain_mm)

def execute_verbose(dest,binary):
    print(f'Running {dest.name}…',flush=True)
    try:
        ws.execute(dest,binary)
    except Exception:
        log=dest/'model.log'
        print('\n===== NATIVE WATEM/SEDEM LOG =====',flush=True)
        if log.exists(): print(log.read_text(),flush=True)
        else: print('No executable log was written.')
        print('\n===== NATIVE CONFIGURATION =====',flush=True)
        ini=dest/'modelinput/inifile.ini'
        if ini.exists(): print(ini.read_text(),flush=True)
        print('\n===== MODEL INPUT FILES =====',flush=True)
        for p in sorted((dest/'modelinput').glob('*')):
            if p.is_file(): print(p.name,p.stat().st_size,'bytes')
        raise
    print(f'{dest.name}: completed',flush=True)

assert set(scenarios)=={'annual','event'}, f'Block 5 did not finish preparing both scenarios. Available: {list(scenarios)}. Rerun block 5 and inspect its first error before running block 6.'
for mode in ('annual','event'):
    assert (scenarios[mode].sfolder.scenario_folder/'modelinput'/'inifile.ini').is_file(), f'Missing native {mode} inputs; rerun block 5.'
BASELINE=SITE/'baseline'; BASELINE.mkdir(exist_ok=True)
# Preserve failed runs for diagnosis; start a clean baseline on rerunning this cell.
if any(BASELINE.iterdir()):
    import uuid
    archived=SITE/('baseline_previous_'+uuid.uuid4().hex[:8])
    BASELINE.rename(archived)
    BASELINE.mkdir()
    print('Previous baseline preserved at',archived,flush=True)
rows=[]
for mode in ('annual','event'):
    s=scenarios[mode]
    dest=BASELINE/('annual' if mode=='annual' else 'event_40mm')
    ws.clone_run(s.sfolder.scenario_folder,dest,binary,40 if mode=='event' else None)
    execute_verbose(dest,binary)
    rows.append(report_soil(dest,dest.name,40 if mode=='event' else None))
display(pd.DataFrame(rows)); pd.DataFrame(rows).to_csv(BASELINE/'summary.csv',index=False)

# --- Compact model atlas: actual native files, common basin extent ---
from matplotlib.collections import LineCollection
from matplotlib.colors import SymLogNorm
import matplotlib.colors as colors
from matplotlib.patches import Patch
from IPython.display import display
def model_atlas(folder, inputs=False):
    inp=folder/'modelinput'; out=folder/'modeloutput'; report=folder/'report'
    cfg=ws.read_ini(inp/'inifile.ini')
    prc,prof=ws.read_raster(ws.raster_path(cfg,inp,'Parcel filename'))
    active=prc!=0; channel=prc==-1; res=abs(prof['transform'].a)
    def native(path):
        with rasterio.open(path) as ds:
            assert ds.shape==prc.shape and ds.transform.almost_equals(prof['transform']),path
            a=ds.read(1).astype(float)
            valid=active & np.isfinite(a) & (a!=-9999) & (a!=-99999)
            if ds.nodata is not None and ds.nodata!=0: valid &= a!=ds.nodata
            return a,valid
    specs=[]
    def add(path,title,unit,cmap='viridis',scale=1,kind='normal',export=None):
        if not path.is_file(): return
        a,v=native(path)
        a*=scale
        specs.append((title,unit,a,v,cmap,kind))
        if export: ws.tif_write(report/(export+'.tif'),a,prof,v)
    if inputs:
        known={'dtm filename':('Conditioned elevation','m','terrain',1,'normal'),'parcel filename':('Composite land cover','class','tab20',1,'prc'),'c factor map filename':('C factor','–','YlOrRd',1,'c'),'k factor filename':('K factor (native units)','–','YlOrBr',1,'normal'),'p factor map filename':('P factor','–','viridis',1,'normal'),'ktc map filename':('Transport coefficient kTC','native units','magma',1,'normal'),'cn map filename':('Curve Number','–','YlGnBu',1,'normal'),'shapefile catchment':('Watershed mask','class','Greens',1,'normal'),'outlet map filename':('Explicit outlet','ID','Blues',1,'normal')}
        for key,(label,unit,cmap,scale,kind) in known.items():
            try: pth=ws.raster_path(cfg,inp,key)
            except AssertionError: continue
            add(pth,label,unit,cmap,scale,kind)
        specs.append(('Model river network','class',channel.astype(float),active,'Blues','normal'))
    else:
        known=[('SLOPE.rst','Slope','degrees','terrain',180/np.pi,'normal','slope_deg'),('AspectMap.rst','Aspect (native radians)','rad','twilight',1,'normal','aspect_rad'),('UPAREA.rst','Effective contributing area','ha','viridis',.0001,'log','effective_uparea_ha'),('LS.rst','LS factor','–','magma',1,'normal','LS'),('RUSLE.rst','Potential soil loss','t/ha','YlOrRd',10,'normal','potential_erosion_t_ha'),('Capacity.rst','Transport capacity','kg/cell','magma',1,'normal','capacity_native'),('WATEREROS (kg per gridcel).rst','Net redistribution (negative = erosion)','t/ha',redistribution_cmap,10/res**2,'signed','erosion_t_ha'),('WATEREROS (mm per gridcel).rst','Net surface change','mm',redistribution_cmap,1,'signed','net_surface_change_mm'),('SediIn_kg.rst','Sediment entering cell','t/cell','magma',.001,'log','sediment_in_t'),('SediOut_kg.rst','Sediment leaving cell','t/cell','magma',.001,'log','sediment_out_t'),('SediExport_kg.rst','Sediment export','t/cell','magma',.001,'log','sediment_export_t'),('Remap.rst','Rainfall excess / deficit','mm','BrBG',1,'signed','rainfall_excess_signed_mm'),('Total runoff.rst','Cumulative runoff reaching cell','m³/cell','Blues',1,'log','total_runoff_m3')]
        for name,title,unit,cmap,scale,kind,export in known: add(out/name,title,unit,cmap,scale,kind,export)
        remap=out/'Remap.rst'
        if remap.exists():
            a,v=native(remap); specs.append(('Local runoff generation','mm',np.maximum(a,0),v,'Blues','normal'))
    if not specs: return
    cols=3; nr=(len(specs)+cols-1)//cols
    fig,axes=plt.subplots(nr,cols,figsize=(15,4.1*nr),squeeze=False,constrained_layout=True)
    for ax,(title,unit,a,v,cmap,kind) in zip(axes.flat,specs):
        ax.set_title(title,fontsize=11); ax.set_axis_off()
        if not v.any(): ax.text(.5,.5,'No valid data',ha='center'); continue
        if kind=='prc': plot_classes(ax,prc,active,title=title); continue
        if kind=='c': plot_cfactor(ax,a,active,title); continue
        values=a[v]; norm=None
        if kind=='signed':
            lim=max(float(np.percentile(np.abs(values),98)),1e-8); norm=colors.Normalize(-lim,lim)
        elif kind=='log':
            lim=max(float(np.max(values)),1e-8); norm=SymLogNorm(linthresh=max(lim/1000,1e-8),vmin=0,vmax=lim)
        im=ax.imshow(np.ma.masked_where(~v,a),cmap=cmap,norm=norm,interpolation='nearest')
        if channel.any() and title!='Model river network': ax.contour(channel,levels=[.5],colors='#196da5',linewidths=.25)
        fig.colorbar(im,ax=ax,shrink=.72,label=unit)
    for ax in list(axes.flat)[len(specs):]: ax.set_axis_off()
    fig.suptitle(('MODEL INPUTS' if inputs else 'MODEL OUTPUTS')+' — '+folder.name,fontsize=15)
    fig.savefig(report/('input_atlas.png' if inputs else 'output_atlas.png'),dpi=150,bbox_inches='tight'); plt.show(); plt.close(fig)
    inventory=[]
    for pth in sorted((inp if inputs else out).iterdir()):
        if pth.is_file() and pth.suffix.lower() in ['.rst','.txt','.ini']:
            inventory.append({'file':pth.name,'bytes':pth.stat().st_size})
    pd.DataFrame(inventory).to_csv(report/('input_inventory.csv' if inputs else 'output_inventory.csv'),index=False)

def routing_atlas(folder):
    inp=folder/'modelinput'; out=folder/'modeloutput'; report=folder/'report'
    cfg=ws.read_ini(inp/'inifile.ini'); prc,prof=ws.read_raster(ws.raster_path(cfg,inp,'Parcel filename'))
    route=out/'routing.txt'
    if not route.exists(): print('Routing table not written:',folder.name); return
    t=pd.read_csv(route,sep='\t'); t.columns=t.columns.str.strip().str.lower()
    for k in t: t[k]=pd.to_numeric(t[k].astype(str).str.replace(',','.',regex=False),errors='raise')
    records=[]; shape=prc.shape
    for j in (1,2):
        for a,b,c,d,f in zip(t.row,t.col,t[f'target{j}row'],t[f'target{j}col'],t[f'part{j}']):
            a,b,c,d=map(int,(a-1,b-1,c-1,d-1))
            if f<=0 or not(0<=a<shape[0] and 0<=b<shape[1] and 0<=c<shape[0] and 0<=d<shape[1]): continue
            if prc[a,b]==0: continue
            records.append((a,b,c,d,float(f),prc[a,b]==-1 and prc[c,d]==-1))
    routes=pd.DataFrame(records,columns=['from_row0','from_col0','target_row0','target_col0','fraction','channel_to_channel'])
    routes.to_csv(report/'native_routing_links.csv',index=False)
    ch=routes[routes.channel_to_channel]
    if folder.name.startswith('event') and ROUTE_CHANNEL_RUNOFF:
        expected=pd.read_csv(PACKAGE/'channel_routing_links.csv')
        keys=['from_row0','from_col0','target_row0','target_col0']
        assert set(map(tuple,expected[keys].to_numpy()))==set(map(tuple,ch[keys].to_numpy())),'Native channel routing differs from D8 links'
        assert np.allclose(ch.fraction,1),'Channel fractions differ from one'
    fig,ax=plt.subplots(figsize=(9,8),constrained_layout=True)
    ax.imshow(np.ma.masked_where(prc==0,np.ones(shape)),cmap='Greys',vmin=0,vmax=3)
    for is_ch,color,width in [(False,'#777777',.25),(True,'#0064be',1.5)]:
        sub=routes[routes.channel_to_channel==is_ch]
        if len(sub):
            ax.add_collection(LineCollection([[(b,a),(d,c)] for a,b,c,d in sub.iloc[:,:4].to_numpy()],colors=color,linewidths=width,alpha=.8))
            if is_ch:
                ax.quiver(sub.from_col0,sub.from_row0,sub.target_col0-sub.from_col0,sub.target_row0-sub.from_row0,angles='xy',scale_units='xy',scale=1,color=color,width=.002)
    outlet_path=inp/'explicit_outlet.rst' if (inp/'explicit_outlet.rst').exists() else inp/'Outlet.rst'
    if outlet_path.exists():
        with rasterio.open(outlet_path) as ds: oa=ds.read(1)
        rr,cc=np.where(oa>0); ax.scatter(cc,rr,c='red',s=45,label='Native outlet',zorder=5); ax.legend()
    ax.set_title('Native routing — grey: hillslope; blue arrows: channel'); ax.set_axis_off()
    fig.savefig(report/'routing_atlas.png',dpi=180); plt.show(); plt.close(fig)
    print(folder.name,':',len(routes),'routing links;',len(ch),'channel-to-channel links')
    if folder.name.startswith('event') and (out/'Total runoff.rst').exists():
        print('Total runoff is cumulative water reaching each cell (m³); never sum across cells as a catchment water budget.')

# Show inputs once (40 mm includes CN and outlet), annual outputs once, and
# separate event output maps using the same catchment grid.
model_atlas(BASELINE/'event_40mm',inputs=True)
for mode in ['annual','event_40mm']:
    model_atlas(BASELINE/mode)
routing_atlas(BASELINE/'event_40mm')
# Save a simple forcing and response comparison.
fig,axes=plt.subplots(2,1,figsize=(10,7),constrained_layout=True)
for depth in [40]:
    mode=f'event_{depth}mm'; f=BASELINE/mode
    rain_cfg=ws.read_ini(f/'modelinput/inifile.ini')
    rain_data=np.loadtxt(ws.raster_path(rain_cfg,f/'modelinput','Rainfall filename'))
    h=pd.read_csv(f/'report/hydrograph.csv')
    axes[0].step(rain_data[:,0],rain_data[:,1],where='pre',label=mode)
    axes[1].plot(h.time_min,h.discharge_m3_s,label=mode)
axes[0].set(xlabel='Time (min)',ylabel='Rainfall per interval (mm)')
axes[1].set(xlabel='Time (min)',ylabel='Discharge (m³/s)')
for ax in axes: ax.legend(); ax.grid(alpha=.2)
fig.savefig(PACKAGE/'rainfall_hydrographs.png',dpi=150); plt.show(); plt.close(fig)

(BASELINE/'complete.json').write_text(json.dumps(dict(kind='baseline',baseline_id=ws.baseline_digest(BASELINE),site=SITE_ID,year=YEAR,resolution=RES,storms=[40],parameter_status='provisional K=30, HSG B, annual C used for event',cn_alpha=CN_ALPHA,cn_beta=CN_BETA,cn_ar5_mm=CN_AR5_MM,channel_runoff_routing=ROUTE_CHANNEL_RUNOFF,reference=str(PACKAGE)),indent=2))


## 7. Export the portable GIS package
Save the ZIP locally. It contains source and derived GIS layers, data lineage and parameter records, verified native model inputs, baseline results and empty scenario templates for NB02.

In [ ]:
#@title 7 — Download GIS and model package
# Empty shapefile templates with explicit schema, readable in ArcGIS and QGIS.
try:
    import fiona
except ModuleNotFoundError:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','fiona'])
    import fiona
from google.colab import files
# Validate a complete baseline before touching the export staging directory.
modes=['annual','event_40mm']
def complete_baseline(p):
    if not p.is_dir() or not (p/'complete.json').is_file(): return False
    try:
        meta=json.loads((p/'complete.json').read_text())
        return meta.get('kind')=='baseline' and meta.get('site')==SITE_ID and all(
            (p/m/'report'/'summary.csv').is_file() and
            (p/m/'modelinput'/'inifile.ini').is_file() for m in modes
        ) and meta.get('baseline_id')==ws.baseline_digest(p)
    except (OSError,ValueError): return False
if not complete_baseline(BASELINE):
    candidates=[p for p in SITE.glob('baseline*') if complete_baseline(p)]
    assert len(candidates)==1, f'Expected one recoverable complete baseline, found {candidates}. Inspect block 6; do not export incomplete runs.'
    BASELINE=candidates[0]
    print('Recovered verified baseline:',BASELINE)
# Use fresh staging on every export to avoid stale GIS files or copytree collisions.
import tempfile
GIS=Path(tempfile.mkdtemp(prefix='GIS_baseline_',dir=WORK))
for sub in ['reference','baseline_results','editable_scenario']: (GIS/sub).mkdir()
for f in PACKAGE.iterdir():
    if f.is_file() and f.suffix.lower() in ['.shp','.shx','.dbf','.prj','.cpg','.gpkg','.tif','.csv','.json','.png','.ini','.txt']: shutil.copy2(f,GIS/'reference'/f.name)
for mode in modes: shutil.copytree(BASELINE/mode/'report',GIS/'baseline_results'/mode)
schemas={'C_change':('Polygon',{'id':'int','c_mult':'float','practice':'str:80','years':'int','comment':'str:200'}),'CN_change':('Polygon',{'id':'int','cn_delta':'float','soc_base':'float','soc_new':'float','years':'int','practice':'str:80','comment':'str:200'}),'grass':('LineString',{'id':'int','width_m':'float','type':'str:40','comment':'str:200'})}
for name,(geom,props) in schemas.items():
    with fiona.open(GIS/'editable_scenario'/f'{name}.shp','w',driver='ESRI Shapefile',crs='EPSG:31370',schema={'geometry':geom,'properties':props}): pass
(GIS/'README.txt').write_text('''Open layers in ArcGIS Pro or QGIS. Edit ONLY the three shapefile templates. Draw polygons for C multiplier (0–1) and CN delta (-20 to 0); draw grass centerlines with total width in metres. Save edits; ZIP the editable_scenario folder including SHP, SHX, DBF, PRJ. Upload in NB02. SOC fields document assumptions only. Parameters are provisional; compare relative effects, not validated predictions. River discharge is at the explicit confirmed outlet when channel routing is enabled; otherwise it is the automatic outlet. Annual C is reused for events in this pilot.''')
# A single portable ZIP contains GIS layers and machine-readable native baseline inputs.
shutil.copytree(BASELINE,GIS/'baseline',dirs_exist_ok=True)
archive=shutil.make_archive(str(SITE/f'{SITE_ID}_Baseline'),'zip',GIS)
print('Save this ZIP locally: it is needed for NB02 and NB03.'); files.download(archive)
